# DMFA Lifetime
This notebook contains the DMFA model, and produces the figures for the paper. It shows the inconsistency between common LCA lifetime assumptions and empirical building lifetimes in Quebec.

The results of this notebook are directly influenced by the available data. Due to their large size relative to flows, it's important to have accurate stock data - even small stock errors can have a large influence on inflow and outflow data, since they 'live' on very different scales (Skaberskis, in [CMHC 1993](https://publications.gc.ca/collections/collection_2018/schl-cmhc/nh15/NH15-694-1993-eng.pdf)). As a quick indication, modern annual levels for Quebec are ~3.5M dwellings for the total stock, ~40k for inflows, ~5k for outflows.  

In [ ]:
# Imports
import glob
import math
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
# from matplotlib import gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import Colormap
from numpy.random import PCG64, SeedSequence
from scipy import stats
from scipy.optimize import differential_evolution
from scipy.signal import savgol_filter
from scipy.special import gamma as _gamma

from parqc import dynamic_stock_model as dsm

In [ ]:
# RNG
"""
https://numpy.org/doc/stable/reference/random/index.html
https://numpy.org/doc/stable/reference/random/generator.html

import secrets
secrets.randbits(128)
>>>304553770673275539343018914623208856046"""

RNG_SEED = 304553770673275539343018914623208856046
SIZE = 5000
ss = SeedSequence(RNG_SEED)
print("seed = {}".format(ss.entropy))
bg = PCG64(ss)
rng = np.random.default_rng(bg)

# NOTE Although this seeding approach follows best practice, the internal state of the generator changes with each cell (each consumed random number). Consequently, the notebook is reproductible when run from a fresh kernel, from top to bottom.
# FIXME Consider running independent child streams

In [ ]:
# Model
MODEL_START = 1607
MODEL_END = 2066  # FIXME cut to 2025
LAST_MDL_YR = 2021
MODEL_TIME = range(MODEL_START, MODEL_END + 1)
Nt = len(MODEL_TIME)  # Total years
FUTURE_COHORTS_START = LAST_MDL_YR
FUTURE_COHORTS_END = MODEL_END
TARGET_TYPES = [
    "apartments",
    "mobile",
    "single_attached",
    "single_detached",
]
TARGET_COHORTS = [ # From the 2021 census definition
    "1608-2025",  # changed to 1608-2025, from 0-2025
    "1608-1920",
    "1921-1945",
    "1946-1960",
    "1961-1970",
    "1971-1980",
    "1981-1990",
    "1991-1995",
    "1996-2000",
    "2001-2005",
    "2006-2010",
    "2011-2015",
    "2016-2020",
    "2021-2025",
]

cohort_bins = [
    (1608, 1920, "1608-1920"),
    (1921, 1945, "1921-1945"),
    (1946, 1960, "1946-1960"),
    (1961, 1980, "1961-1980"),
    (1981, 2000, "1981-2000"),
    (2001, 2021, "2001-2021"),
]

# Paths
DATA_DIR = os.path.abspath("../data")
FIG_DIR = os.path.abspath("../figs")
if not os.path.exists(FIG_DIR):
    os.makedirs(FIG_DIR)

# Figures
TIME_WINDOW = (1850, 2021)
SAVE_FIGS = False
COLOR_PALETTE = "colorblind" # or 'Spectral'
W_INCH = 4.5 * 1
H_INCH = 2.5 * 1
DPI = 1000

# Calibration
CALIB_WEIGHTS = (0.1, 0.6, 0.3)  # (0.4, 0.30, 0.30)
RSD = 0.3
BASE_SCENARIOS = [
    {
        "label": f"Normal ($\\mu$={lt})",
        "dist": {"Type": "Normal", "Mean": lt, "StdDev": RSD * lt},
    }
    for lt in [50, 75, 100, 150, 200]
] + [
    {
        "label": "Weibull (Deetman)",
        "dist": {"Type": "Weibull", "Shape": 1.97, "Scale": 57.53},
    },
    {
        "label": "Weibull (Ianchenko)",
        "dist": {"Type": "Weibull", "Shape": 2.629, "Scale": 149.67},
    },
    {
        "label": "LogNormal (Ianchenko)",
        "dist": {"Type": "LogNormal2", "Mean": 4.625, "StdDev": 0.574},
    },
]
SUBSET = [
    "Normal ($\\mu$=100)",
    "Normal ($\\mu$=150)",
    "Normal ($\\mu$=200)",
    "Weibull (Deetman)",
    "Weibull (Ianchenko)",
    "LogNormal (Ianchenko)",
    "Weibull (calibrated)",
    "Normal (calibrated)",
    "LogNormal (calibrated)",
]

In [ ]:
# Utils
def get_colors(cmap, n):
    """Return n colors from a matplotlib cmap, seaborn palette, or color list."""

    # from matplotlib
    if isinstance(cmap, Colormap):
        return [cmap(i / max(n - 1, 1)) for i in range(n)]

    # if string
    if isinstance(cmap, str):
        try:
            mpl_cmap = plt.get_cmap(cmap, n)
            return [mpl_cmap(i) for i in range(n)]
        except ValueError:
            # try seaborn palette name
            import seaborn as sns
            return sns.color_palette(cmap, n)

    # list/tuple of colors
    try:
        if len(cmap) >= n:
            return list(cmap[:n])
    except TypeError:
        pass

    raise ValueError(
        f"Could not interpret cmap={cmap!r}. "
        "Pass a matplotlib cmap, seaborn palette name, or list of colors."
    )

## Importing and cleaning data
### Dwelling datasets (NEUD, StatCan)
Based on [Table 98-10-0233-01  Dwelling condition by tenure: Canada, provinces and territories, census divisions and census subdivisions](https://doi.org/10.25318/9810023301-eng), the confidence interval on dwellings by period of construction is pretty tight (at least for 2021). The confidence interval for the 'total' value is very small (i.e. ~20/3.5M), however for most cohorts it's i ~0.4%. 

In [ ]:
# Import 'raw' census dwelling counts, as prepared by "dataprep.py"
infile = os.path.join(DATA_DIR, "clean", "fulldata.parquet")
census_dw = pd.read_parquet(infile) # uncertimety_data


In [ ]:
# Import datasets from the National Energy Use Database (NEUD)
# TODO: Ajouter uniformisation des cohortes


def import_neud_dataset(unit, data_dir=DATA_DIR):
    """Import and clean the neud datasets

    This function is meant to import and clean the data from tables 16, 17, 19 and 20 of the Comprehensive Energy Use Database (neud):
        - https://oee.nrcan.gc.ca/corporate/statistics/neud/dpa/menus/trends/comprehensive/trends_res_qc.cfm
    The .csv files must be cleaned beforehand to contain only the actual values (housing stock, floor space), not their shares.

    Args:
        unit (str): 'm2' or 'dw'
        data_dir (str, optional): absolute path to data directory. Defaults to DATA_DIR.

    Raises:
        ValueError: if unit

    Returns:
        pd.DataFrame: The data is returned in long format. To convert it, use e.g.: df.pivot(index=['annee', 'vintage'], columns='type', values='stock')
    """
    try:
        infiles = glob.glob(DATA_DIR + "/nb" + "/*" + unit + ".csv")

        if len(infiles) == 0:
            raise ValueError(
                f"Unexpected {unit}: glob returns no files from the following path {DATA_DIR + '/nb' + '/*' + unit + '.csv'}"
            )
            # FIXME: May need to return NameError instead, as if infiles could not be found (NameError: name 'infiles' is not defined)?

        imported_dfs = []
        for infile in infiles:
            df = pd.read_csv(infile, encoding="utf-8", sep=";", decimal=",")

            # Data cleaning
            # Add type and rename columns
            cols = df.columns.to_list()
            dw_type = (
                cols[0]
                .strip()
                .lower()
                .replace(" ", "_")
                .split("_housing_")[0]  # in /*dw.csv files
                .split("_floor_")[0]  # in /*m2.csv files
                .replace("_homes", "")
            )

            df["type"] = dw_type
            cols = ["type"] + cols
            df = df[cols]
            cols[1] = "vintage"
            df.columns = cols

            imported_dfs.append(df)

        # Combiner en un seul dataframe
        combined_df = pd.concat(imported_dfs, ignore_index=True)

        # Retirer les sous-totaux et totaux par type de logement
        combined_df = combined_df.loc[
            ~(combined_df["vintage"].isin(["subtotal", "Total"])), :
        ]

        # Uniformiser les cohortes
        combined_df.loc[combined_df["vintage"] == "Before 1946", "vintage"] = "<1946"

        # Corriger les tirets
        combined_df["vintage"] = [
            _.replace("–", "-") for _ in combined_df["vintage"].values
        ]

        # Convertir (et retourner) le dataframe en format long
        return (
            pd.DataFrame(combined_df.set_index(["type", "vintage"]).stack(level=0))
            .reset_index()
            .rename(columns={"level_2": "year", 0: "stock"})
        )

    except TypeError as err:
        # e.g. import_neud_dataset(23234)
        print(type(err), err)
        return None

    except ValueError as err:
        # e.g. import_neud_dataset('m8990')
        print(type(err), err)
        return None


In [ ]:
# Import NEUD datasets in dwelling units (dw)
neud_dw = import_neud_dataset("dw")

In [ ]:
# Import input dataset, as prepared by interpolate.py
# Long (tidy) dataset, starting with first census data available (1685) 
dataset = pd.read_parquet(
    DATA_DIR + "/clean/reconciled.parquet",
)

In [ ]:
# Wide version of the reconciled, tidy 'dataset', including all model time (1607-2021)
new_dwlgs = (
    dataset[(dataset["vintage"] == "1608-2025") & (dataset["type"] != "total")]
    .copy()
    .pivot(index="census_year", columns="type", values="dwellings")
    .reindex(MODEL_TIME)
)
# Set zero dwellings in 1607
new_dwlgs.loc[1607, :] = 0

# Interpolate forward 1607-1685
new_dwlgs = new_dwlgs.interpolate(method="index", limit_direction="forward").round()

# copy 'total'
total_stock = new_dwlgs.round(0).astype("int").sum(axis=1)

# Show dataset
new_dwlgs.head()

### Historic dwelling flow data
#### Dwelling completions

In [ ]:
# Sources: NH12-1-1957-4.pdf and Statistics Canada / CMHC
# NH12-1-1957-4.pdf also has info on vacancy rates and delays; Statistics Canada on single-detached, multiples, semi-detached and row dwellings.
# Dwelling starts data are also available in Steele's "Section S: Construction and Housing" statistics S181-189, 1948-1976. https://www150.statcan.gc.ca/n1/pub/11-516-x/sections/4057757-eng.htm

nh_data = {
    1950: 27237,
    1951: 26686,
    1952: 22407,
    1953: 29803,
    1954: 26182,
    1955: 34866,
    1956: 41166,
    1957: 33188,
    1958: 39750,
    1959: 38920,
    1960: 31311,
    1961: 31756,
    1962: 35782,
    1963: 38989,
    1964: 43658,
    1965: 42565,
    1966: 40412,
    1967: 39108,
    1968: 38961,
    1969: 44605,
    1970: 36608,
    1971: 48783,
    1972: 53466,
    1973: 55260,
    1974: 58596,
    1975: 51540,
    1976: 54301,
    1977: 61979,
    1978: 54129,
    1979: 44288,
    1980: 33560,
    1981: 30691,
    1982: 21526,
    1983: 35681,
    1984: 43410,
    1985: 41577,
    1986: 56984,
    1987: 68949,
    1988: 65224,
    1989: 50855,
    1990: 52630,
    1991: 42720,
    1992: 42323,
    1993: 34859,
    1994: 36345,
    1995: 23363,
    1996: 22194,
    1997: 26308,
    1998: 22944,
    1999: 24141,
    2000: 23346,
    2001: 26381,
    2002: 36308,
    2003: 45123,
    2004: 52610,
    2005: 49205,
    2006: 48668,
    2007: 48605,
    2008: 47956,
    2009: 43341,
    2010: 45969,
    2011: 44412,
    2012: 45042,
    2013: 39881,
    2014: 39206,
    2015: 33589,
    2016: 37536,
    2017: 40241,
    2018: 42062,
    2019: 44602,
    2020: 46360,
    2021: 53078,
    2022: 57649, # FIXME remove to fit census?
}

#### Dwelling demolitions
Here I use data at the provincial level, but datais available at smaller granularity if required.

In [ ]:
infile = os.path.join(DATA_DIR, "nb", "34100079.csv")
nh_demolished = pd.read_csv(infile, sep=",", usecols=[0, 1, 10])
nh_demolished = nh_demolished[nh_demolished["GEO"] == "Quebec"]

## Preparing the data
### Completing lists (type,cohort)

In [ ]:
# Extract the NEUD dwelling cohorts, and expand to full MODEL_TIME
neud_cohorts = sorted(neud_dw["vintage"].unique().tolist())
neud_cohorts = [neud_cohorts[-1]] + neud_cohorts[:-1]

# Add future cohorts (2021-2066)
neud_cohorts = neud_cohorts + (
    [
        f"{a}-{b}"
        for a, b in zip(
            [i for i in range(FUTURE_COHORTS_START, FUTURE_COHORTS_END, 5)],
            [
                i
                for i in range(
                    5 * (FUTURE_COHORTS_START // 5 + 1),
                    5 * (FUTURE_COHORTS_END // 5),
                    5,
                )
            ]
            + [FUTURE_COHORTS_END],
        )
    ]
)

# The cohorts presented here follow the NEUD defition.
neud_cohorts = {
    i + 1: cohort.replace("–", "-") for i, cohort in enumerate(neud_cohorts)
}

### Typesplit
For older, missing census years, we assume that the urban population follows Montreal city centre (the only available data) and that the rest is 90-10% single detached, similarly to the Montreal parishes.

In [ ]:
# NOTE: Assuming that in 1870, Mtl was 130000 inhabitants, and ~325000 in 1900, this fits with a ~40% urban population in Quebec (prov.) in 1901 (https://statistique.quebec.ca/fr/fichier/retrospective-du-20e-siecle.pdf), and the data found in Stone 1967: ~15% urban in 1850, ~36% in 1901 (https://archive.org/details/1961995421967eng/page/28/mode/2up). Similar source available here: p.55 (https://numerique.banq.qc.ca/patrimoine/details/52327/2828103?docref=0j61nu7o6oGIP66f2AiPeQ)


stone1967 = {  # percent of population urban, canada and provinces, 1851-1961
    1851: 0.149,
    1861: 0.166,
    1871: 0.199,
    1881: 0.238,
    1891: 0.286,
    1901: 0.361,
    1911: 0.445,
    1921: 0.518,
}

mtl_type_share = {  # percent of dwellings single-family
    1851: 0.52,
    1861: 0.52,
    # 1871:.402,
    1881: 0.402,
    # 1891:.314,
    1901: 0.314,
    1911: np.nan,
    1921: np.nan,
}

# Calculate share of single-family dwellings in Quebec, accounting for urban-rural differences
inferred_tsplit = {}
for year, value in mtl_type_share.items():
    inferred_tsplit[year] = np.average(
        [value, 0.9], weights=[stone1967[year], 1 - stone1967[year]]
    )

In [ ]:
# Get the 'total' type rows as the denominator
dwlg_totals = dataset[dataset["type"] == "total"].set_index(["census_year", "vintage"])[
    "dwellings"
]

# Get counts by type (excluding 'total')
dwlg_count = (
    dataset[dataset["type"] != "total"]
    .groupby(by=["census_year", "vintage", "type"])["dwellings"]
    .sum()
)

# Divide - the MultiIndex aligns on year, vintage
dwlg_share = dwlg_count / dwlg_totals


In [ ]:
# Quick check: values != 1
g = dwlg_share.groupby(by=["census_year", "vintage"]).sum()
print(f"There are {len(g[(g > 0) & (g < 1)])} values different from 0 or 1")
g[(g > 0) & (g < 1)].agg(["min", "max"])

# Check isna()
dwlg_share[dwlg_share.isna()]

In [ ]:
# Large plot; can show interesting inversion of apartments/single-attached ca. 1921. Heavy, only run if needed.
run_this = False
if run_this:
    g = sns.relplot(
        pd.DataFrame(dwlg_share),
        x="census_year",
        y="dwellings",
        hue="type",
        col="vintage",
        col_wrap=3,
        # facet_kws={"set_xlim":(1920,2021)}
    )

    g.set(xlim=(1920, 2021))

In [ ]:
# Calculate the stock typesplit
updated_tsplit = (  # previously 'reupdated'
    dataset[dataset["vintage"] == "1608-2025"]
    .pivot(index="census_year", columns="type", values="dwellings")
    .loc[:, TARGET_TYPES]
    .div(
        dataset[dataset["vintage"] == "1608-2025"]
        .pivot(index="census_year", columns="type", values="dwellings")
        .loc[:, "total"]
        .to_numpy()
        .reshape(-1, 1)
    )
).sort_index(axis=1)


df = pd.DataFrame(dwlg_share).reset_index()
type_split_1608 = (
    df[
        df["vintage"] == "1608-2025"
    ]  # NOTE: df relies on dwlg_share, which is adapted from the reconciled dataset
    .pivot(index="census_year", columns="type", values="dwellings")
    .reindex(MODEL_TIME)
    .interpolate(method="index", limit_direction="both")
) # FIXME unnecessary, remove
tsplit = type_split_1608.reindex(MODEL_TIME).ffill().bfill()
tsplit = tsplit[TARGET_TYPES]

# Overwrite updated_tsplit
updated_tsplit.loc[:1931, :] = [np.nan, np.nan, np.nan, np.nan]

updated_tsplit.loc[1931, :] = [
    0.40982717,  # apartments
    np.nan,  # mobile
    0.08130652,  # single attached
    0.50386409,  # single detached
]  # for 1931, 'housing in Canada' donne array([0.50386409, 0.08130652, 0.40982717])

for year, sd_split in inferred_tsplit.items():
    updated_tsplit.loc[year, :] = [
        1 - sd_split,  # apartments
        np.nan,  # mobile
        np.nan,  # single attached
        sd_split,  # single detached
    ]

# NOTE: Hanna and Dufaux recorded that single_attached dwellings were so rare to not warrant inclusion, which might suggest a value (close to) zero in 1851 for single attached and mobile. Currently left as-is since it's so low.
# updated_tsplit.loc[1851, "single_attached"] = 0
# updated_tsplit.loc[1851, "mobile"] = 0

# also fix updated_tsplit
updated_tsplit.interpolate(method="index", limit_direction="forward")
updated_tsplit = updated_tsplit.reindex(MODEL_TIME).interpolate(
    method="index", limit_direction="both"
)

# normalize both dataframes so that each row sums to one
# TODO Create unittest: e.g., tsplit[tsplit.sum(axis=1) != 1.0].sum(axis=1)
tsplit = tsplit.div(tsplit.sum(axis=1).squeeze(), axis=0)
updated_tsplit = updated_tsplit.div(updated_tsplit.sum(axis=1).squeeze(), axis=0)

fig, ax = plt.subplots(2, 2, figsize=(10, 10), dpi=150)
ax = ax.flatten()
tsplit.plot(ax=ax[0], legend=False)
tsplit.plot(kind="area", ax=ax[1], legend=False)

updated_tsplit.plot(ax=ax[2], legend=False)
updated_tsplit.plot(kind="area", ax=ax[3], legend=False)

plt.tight_layout()
plt.show()

In [ ]:
# Smoothed stock typesplit for a sensitivity analysis
smooth_typesplit = updated_tsplit.copy()

# Savitzky-Golay filter
window_length = 31  # NOTE this was adjusted manually, other values might work similarly. This implies an assumption that dwelling type shares evolve smoothly over ~31 years. Tests with 11, 15, and 21 years showed no noticeable impacts on type split trends.
polyorder = 2
smooth_typesplit = smooth_typesplit.apply(
    lambda col: savgol_filter(col, window_length=window_length, polyorder=polyorder)
).round(decimals=3)  # NOTE: this removes small negative values resulting from filtering

# rescale to one
smooth_typesplit = smooth_typesplit.div(smooth_typesplit.sum(axis=1), axis=0)

# Compare original vs smoothed
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax = ax.flatten()
updated_tsplit.plot(ax=ax[0], title="Original")
smooth_typesplit.plot(ax=ax[1], title="Smoothed (Savitzky-Golay)")
updated_tsplit.plot(kind="area", ax=ax[2], title="Original (Area)")
smooth_typesplit.plot(kind="area", ax=ax[3], title="Smoothed (Area)")
plt.tight_layout()
if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "smoothed_typesplit.png")
    plt.savefig(file_path, dpi=DPI)
plt.show()

In [ ]:
# make sure the columns order is correct (it already should be)
updated_tsplit = updated_tsplit.sort_index(axis=1)

# Sanity checks

In [ ]:
# Whats the actual error between my calculation and the census data?
# Prepare wide dataset with only census years (1685-2021); all target types and total

# TODO attempt err_cs using census_dw (fulldata)? however, there will be known inaccuracies due to different dwelling type aggregations and missing data.
census_data = (
    dataset[(dataset["vintage"] != "1608-2025")]
    .groupby(["census_year", "type"])
    .sum(numeric_only=True)
    .unstack("type")
)

err_cs = census_data.loc[:, census_data.columns.get_level_values(1) == "total"].copy()

In [ ]:
# Sanity check: check total counts (original, summed) from the datasets
palette = sns.color_palette(COLOR_PALETTE, 5)

sns.set_theme(context="notebook", style="whitegrid", palette=palette)

fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), dpi=150, sharex=True, sharey=False)
axs = axs.flatten()
new_dwlgs.plot(ax=axs[0], color=palette, legend=False)

# add 'total' count
total_stock.plot(ax=axs[0], color=palette[-1], label="total")  # color='black'


# plot stock data from IPFN reconciled dataset
dataset[dataset["vintage"] != "1608-2025"].groupby(["census_year", "type"]).sum(
    numeric_only=True
).unstack("type").plot(style="+", ax=axs[0], color=palette, legend=False)
# NOTE: here, it's ESSENTIAL to filter out the 'total' vintage of '1608-2025'
# however, here, of course it's the same; new_dwlgs == dataset

# handles, labels = ax.get_legend_handles_labels()
# fig.legend(h,l)
axs[0].set_title("Dwelling stocks")
axs[0].set_xlim(*TIME_WINDOW)

axs[1].set_title("Relative error")

# calculate both absolute and relative error
abs_err = -err_cs + total_stock.reindex(err_cs.index).to_numpy().reshape(
    -1, 1
)
rel_err = (abs_err / err_cs) * 100

# plot relative error
rel_err.plot(style=".", color=palette[-1], ax=axs[1], legend=False)
axs[1].hlines(
    [0], min(err_cs.index), max(err_cs.index), colors=["black"], linestyles=["dashed"]
)

# titles
axs[0].set_ylabel("Dwellings")
axs[1].set_ylabel("%")
handles, labels = axs[0].get_legend_handles_labels()
labels = [s.replace("_", " ") for s in labels]
fig.legend(
    handles[0:5],
    labels[0:5],
    loc="upper center",
    bbox_to_anchor=[0.5, 0],
    ncols=3,
    frameon=False,
)

plt.tight_layout(h_pad=0.1)
plt.show()

Expectedly, the stocks fit perfectly, as they're the *same* dataset. Still useful to calibrate lifetime with known flows.

In [ ]:
dataset[dataset["type"].isin(TARGET_TYPES)].groupby(["vintage", "census_year"]).sum(
    numeric_only=True
).unstack("vintage").replace(0, np.nan).interpolate(method="linear").fillna(0).plot(
    legend=False
)  # interpolate: 'ffill'?
plt.xlim(1950, 2025)
plt.ylim(0e6, 1e6)

display(
    dataset[dataset["type"].isin(TARGET_TYPES)]
    .groupby(["vintage", "census_year"])
    .sum(numeric_only=True)
    .unstack("vintage")
)

In [ ]:
dataset["census_year"] = dataset["census_year"].astype("int")
dataset[dataset["type"].isin(TARGET_TYPES)].groupby(["vintage", "census_year"]).sum(
    numeric_only=True
).unstack("vintage").filter(census_dw['census_year'].unique(), axis=0)

### Testing nh data
As a test, can I reproduce data from the censuses using the yearly inflows and outflows? Or at least, does the stock change fit with the observed stock change in my model? 

In [ ]:
# As a test, I reproduce data from the censuses using the yearly inflows and outflows. Does the empirical stock change fit with the modelled stock change?
# nh_data  # covers 1950-2022
# nh_demolished  # covers 1968-1999

# Stock change, 1968-1999
df_a = nh_demolished.drop(columns=["GEO"]).set_index("REF_DATE") # FIXME equal to outflow_data
df_a.index.name = None
df_b = pd.DataFrame.from_dict(nh_data, orient="index", columns=["VALUE"]) # FIXME equal to inflow_data

fig, ax = plt.subplots()
df_stock_change = df_b.reindex_like(df_a) - df_a
df_stock_change.plot(style=".", ax=ax)
ax.set_title("Stock change, 1968-1999")

## Stock change
In stock-driven models:
I(t) = ΔS(t) + O(t)

Based on available empirical datasets, this should hold by definition. Let's confirm it.

In [ ]:
# empirical datasets
inflow_data = pd.Series(nh_data)
outflow_data = nh_demolished.set_index("REF_DATE")["VALUE"]
inflow_data.index.name = "census_year"

# census stock change
stock_change = total_stock.diff().loc[inflow_data.index] # 1950-2022

# calculate the implied outflow, I(t) - Delta_S(t)
implied_outflow = inflow_data - stock_change
residual = implied_outflow[outflow_data.index] - outflow_data
residual.plot()  # kind='scatter'

Having positive residuals (more implied outflows than observed outflows) would suggest that there are "missing" outflows - dwellings leaving through other routes, besides recorded demolitions (under-reporting, conversions, etc.). This would tend to shorten lifetimes based on inflows, and lengthen lifetimes based on outflows.

Since target_stock is interpolated between census counts, ΔS is smooth between census_years, but can introduce 'kinks' on census years.

In [ ]:
fix, ax = plt.subplots(dpi=150)
# expected empirical stock growth
(inflow_data[outflow_data.index] - outflow_data).plot(ax=ax)
# observed stock growth
stock_change[outflow_data.index].plot(ax=ax)
plt.tight_layout()
plt.show()

The recorded historic demolitions (blue) sometimes exceed what the inflow data and stock change can produce (orange line). The modelled stock change (orange) is on average higher than what the empirical inflow data and outflow data suggest (blue, I_obs-O_obs). This could suggest ~5–6 k/yr of dwellings enter the stock through unexpected routes (perhaps conversions/subdivisions, or due to changes and revisions to census methods). Most likely, it is an artifact from data preparation. Contrarily to the real data peaks and valleys (blue), the orange line (ΔS of target_stock) is a step function with breaks at census years (1971, 1976, 1981, 1986, 1991, 1996).

This might affect the calibration process, as the interpolation error tends to overestimate stock change by an amount roughly the same order of magnitude as outflow data.

Also, recorded demolitions should be treated as a lower bound on 'true'  dwelling losses.

In [ ]:
# check stock fit, starting from first available census (1685)
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5), dpi=300)
ax = ax.flatten()
dataset[
    (dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] != "1608-2025")
].astype({"census_year": "int"}).groupby(["census_year", "vintage"])[
    "dwellings"
].sum().unstack("vintage").plot(kind="bar", stacked=True, ax=ax[0])  # plot

# census data
(
    census_dw[
        (census_dw["type"].isin(TARGET_TYPES + ["total"]))
        & (census_dw["vintage"] == "1608-2025")
    ].pivot(index=["census_year"], columns="type", values="dwellings")
).loc[:, "total"].plot(style="o", ax=ax[1])

# compare to 'only' totals, instead of the census_dw values
dataset[
    (dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] == "1608-2025")
].astype({"census_year": "int"}).groupby(["census_year", "vintage"])[
    "dwellings"
].sum().unstack("vintage").loc[:, ["1608-2025"]].plot(
    style=".", color="r", ax=ax[1], legend=False
)  # plot

sns.move_legend(ax[0], loc="upper left", bbox_to_anchor=[0, -0.1], ncols=4)

NOTE: this is a stock *description*. Since it's a stock-driven model, the stock is an input---by definition, it fits! The figure is still relevant as it shows 'initial' data from census_dw, and reconciled data.

In [ ]:
t_df = ( # FIXME somewhat duplicated reltive to total_stock. total_stock covers 1607-2066 since it comes from new_dwlgs. Kept for retrocompatibility for now
    dataset[(dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] == "1608-2025")]
    .astype({"census_year": "int"})
    .groupby(["census_year", "vintage"])["dwellings"]
    .sum()
    .unstack("vintage")
    .loc[:, ["1608-2025"]]
)

intercensal_t = [
    (int(min_yr), int(max_yr))
    for min_yr, max_yr in zip(t_df.index[:-1].to_list(), t_df.index[1:].to_list())
]

test_stock_change = {}
for min_yr, max_yr in intercensal_t:
    # only target relevant cohorts in the loop; anything before 1968 and after 1999 is useless, as I have no 'hard' data for inflows and outflows
    if (min_yr < min(df_stock_change.index)) or (max_yr > max(df_stock_change.index)):
        continue

    print(f"Running for: {min_yr}-{max_yr}")
    # Manually sum the stock change
    m_stock_change = df_stock_change.loc[min_yr:max_yr, :].sum()

    # Calculate the stock difference based on census
    cs_stock_change = t_df.loc[max_yr, :] - t_df.loc[min_yr, :]

    test_stock_change["-".join([str(min_yr), str(max_yr)])] = [
        m_stock_change.to_numpy()[0],
        cs_stock_change.to_numpy()[0],
    ]

# Manually add 1981-1991
test_stock_change["-".join(["1981", "1991"])] = [
    df_stock_change.loc[1981:1991, :].sum().to_numpy()[0],
    (t_df.loc[1991, :] - t_df.loc[1981, :]).to_numpy()[0],
]

# convert to dataframe
test_stock_change = pd.DataFrame.from_dict(
    test_stock_change, orient="index", columns=["manual", "census"]
).sort_index(axis=0)

# plot, both with and without 1986 (obviously wrong value)
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax = ax.flatten()
test_stock_change.plot(style="o", ax=ax[0])
test_stock_change.drop(["1981-1986", "1986-1991"], axis=0).plot(style="o", ax=ax[1])
plt.tight_layout()
plt.show()

In [ ]:
100 * (
    np.abs(
        test_stock_change.drop(["1981-1986", "1986-1991"], axis=0)["manual"]
        - test_stock_change.drop(["1981-1986", "1986-1991"], axis=0)["census"]
    )
    / test_stock_change.drop(["1981-1986", "1986-1991"], axis=0)["census"]
)

That's pretty good. This seems to confirm that the inflow/outflow data agree with the census stock data, and thus that using inflows/outflows as a comparison is a reasonable approach.

The small differences are expected: by definition, dwellings built in a year are counted in that year's stock in the census. However, for 'switch' years, this may not be true, as some dwellings built (early) in 1971 would be counted in the 1971 census, and the rest would be counted in the 1972 census.

This means that in the manual approach, I include all flows that occur in the min and max years (e.g. 1971-1981). However, in the census approach, I leave out some flows late in min_yr and the max_yr, since the census stops in ~may. 

# Final figures
## Skew

In [ ]:
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "0.25",
        "axes.linewidth": 0.8,
        "xtick.color": "0.25",
        "ytick.color": "0.25",
        "figure.dpi": 150,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(W_INCH, H_INCH), sharey=True)


# ----- Helper: Beta PDF -----
def beta_pdf(x, a, b):
    B = math.gamma(a) * math.gamma(b) / math.gamma(a + b)
    return (x ** (a - 1)) * ((1 - x) ** (b - 1)) / B


# Numeric median via CDF inversion (simple trapezoidal integration)
def beta_median(x, pdf):
    dx = np.diff(x)
    trap = (pdf[:-1] + pdf[1:]) * 0.5 * dx
    cdf = np.concatenate([[0], np.cumsum(trap)])
    idx = np.searchsorted(cdf, 0.5)
    if idx == 0:
        return x[0]
    if idx >= len(x):
        return x[-1]
    x0, x1 = x[idx - 1], x[idx]
    c0, c1 = cdf[idx - 1], cdf[idx]
    t = (0.5 - c0) / (c1 - c0)
    return x0 + t * (x1 - x0)


# Common domain for Beta-based shapes
xs = np.linspace(0, 1, 1200)

# Distributions
# Symmetric: Beta(3,3)
a_sym, b_sym = 3, 3
beta_sym = beta_pdf(xs, a_sym, b_sym)
# Positive skew: Beta(2,6)
a_pos, b_pos = 2, 6
beta_pos = beta_pdf(xs, a_pos, b_pos)

# Negative skew: Beta(6,2)
a_neg, b_neg = 6, 2
beta_neg = beta_pdf(xs, a_neg, b_neg)

# Normalize each to unit peak height for visual comparability
beta_sym_n = beta_sym / beta_sym.max()
beta_pos_n = beta_pos / beta_pos.max()
beta_neg_n = beta_neg / beta_neg.max()

# ----- Statistics -----
# Symmetric Beta(3,3): mean = 0.5, mode = (a-1)/(a+b-2) = 0.5, median = 0.5
mean_sym = a_sym / (a_sym + b_sym)
mode_sym = (a_sym - 1) / (a_sym + b_sym - 2)
median_sym = 0.5

# Positive skew Beta(2,6): mean = 0.25; mode = 1/6 ≈ 0.1667; median numeric
mean_pos = a_pos / (a_pos + b_pos)
mode_pos = (a_pos - 1) / (a_pos + b_pos - 2)
median_pos = beta_median(xs, beta_pos)

# Positive skew Beta(2,6): mean = 0.25; mode = 1/6 ≈ 0.1667; median numeric
mean_neg = a_neg / (a_neg + b_neg)
mode_neg = (a_neg - 1) / (a_neg + b_neg - 2)
median_neg = beta_median(xs, beta_neg)

# ----- Panel (a): Symmetric -----
ax0 = axes[0]
color_sym = "0.35"
ax0.plot(
    xs, beta_sym_n, color=color_sym, linewidth=1.6, linestyle="-", label="Symmetric"
)
ax0.text(
    0.02,
    0.98,
    "(a)",
    transform=ax0.transAxes,
    ha="left",
    va="top",
    weight="bold",
    color="0.25",
)
ax0.set_title("Symmetric", pad=2)
ax0.set_xticks([])
ax0.set_yticks([])
ax0.set_xlim(0, 1)

# Vlines: mean, median, mode
for xstat, ls in [(mean_sym, "--"), (median_sym, "-"), (mode_sym, ":")]:
    ax0.axvline(xstat, color=color_sym, linestyle=ls, linewidth=1.0)

# ----- Panel (b): Positive skew -----
ax1 = axes[1]
color_pos = "0.1"
color_neg = color_sym
# Positive skew: solid bold + subtle fill for salience
ax1.plot(
    xs, beta_pos_n, color=color_pos, linewidth=2.2, linestyle="-", label="Positive skew"
)
ax1.fill_between(xs, 0, beta_pos_n, color="0.85", alpha=0.55)
# Negative skew: dotted, same shade as symmetric
ax1.plot(
    xs,
    beta_neg_n,
    color=color_neg,
    linewidth=1.8,
    linestyle="-.",
    label="Negative skew",
)

ax1.set_xticks([])
ax1.set_yticks([])
ax1.set_title("Skewed distributions", pad=2)
ax1.text(
    0.02,
    0.98,
    "(b)",
    transform=ax1.transAxes,
    ha="left",
    va="top",
    weight="bold",
    color="0.25",
)
ax1.set_xlim(0, 1)

# vlines
for xstat, ls in [
    (mean_pos, "--"),
    (median_pos, "-"),
    (mode_pos, ":"),
    (mean_neg, "--"),
    (median_neg, "-"),
    (mode_neg, ":"),
]:
    ax1.axvline(xstat, color=color_pos, linestyle=ls, linewidth=1.0)

# Shared y-limits (normalized peak = 1)
for ax in axes:
    ax.set_ylim(0, 1.05)


# ----- Single combined figure-level legend (ncol=3) -----
# Build handles+labels explicitly: first two for PDFs, followed by three for stats
handles = [
    Line2D([0], [0], color=color_sym, linestyle="-", linewidth=1.6),
    Line2D([0], [0], color=color_pos, linestyle="-", linewidth=2.2),
    Line2D([0], [0], color=color_neg, linestyle="-.", linewidth=1.8),
    Line2D([0], [0], color="0.2", linestyle="--", linewidth=1.2),
    Line2D([0], [0], color="0.2", linestyle="-", linewidth=1.2),
    Line2D([0], [0], color="0.2", linestyle=":", linewidth=1.2),
]
labels = ["Symmetric", "Positive-skewed", "Negative-skewed", "mean", "median", "mode"]

fig.tight_layout(rect=[0, 0.15, 1, 1])
fig.legend(handles=handles, labels=labels, loc="lower center", ncol=3, frameon=False)

# Save
if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "Figure_1.png")
    fig.savefig(file_path, dpi=DPI)
plt.show()

## Probability distribution functions

In [ ]:
sns.set_theme(context="paper", style="whitegrid", palette=COLOR_PALETTE)

# Set up the figure and subplots
fig, axs = plt.subplots(2, 1, figsize=(7, 7), dpi=150, sharex=True, sharey=False)
axs = axs.flatten()

# X range for Normal, Weibull and Lognormal distributions
x1 = np.linspace(0, 300, 1000)

palette = sns.color_palette(COLOR_PALETTE, 8)

# 1. Plotting 5 Normal Distributions in axs[0]
means = [50, 75, 100, 150, 200]
std_devs = [RSD * mean for mean in means]
colors = palette[0:5]  # ['b', 'g', 'r', 'c', 'm']

for mean, std, color in zip(means, std_devs, colors):
    axs[0].plot(
        x1,
        stats.norm.pdf(x1, mean, std),
        label=f"Normal (μ={mean}, σ={std:.2f})",
        color=color,
    )

axs[0].set_title("Normal Distributions")
axs[0].set_xlabel("Years")
axs[0].set_ylabel("Density")

# 2. Plotting Weibull and Lognormal in axs[1]
# Weibull distributions
shape_1, scale_1 = 1.97, 57.53  # Deetman et al. 2019
shape_2, scale_2 = 2.629419, 149.673961  # Ianchenko et al. 2020
axs[1].plot(
    x1,
    stats.weibull_min.pdf(x1, shape_1, scale=scale_1),
    label=f"Weibull (k={np.round(shape_1, 2)}, λ={np.round(scale_1, 2)})",
    color=palette[5],
)
axs[1].plot(
    x1,
    stats.weibull_min.pdf(x1, shape_2, scale=scale_2),
    label=f"Weibull (k={np.round(shape_2, 2)}, λ={np.round(scale_2, 2)})",
    color=palette[6],
)

# LogNormal distribution
mu, sigma = 4.6247768, 0.5737083
axs[1].plot(
    x1,
    stats.lognorm.pdf(x1, sigma, scale=np.exp(mu)),
    label=f"LogNormal (μ={np.round(mu, 2)}, σ={np.round(sigma, 2)})",
    color=palette[7],
)

axs[1].set_title("Weibull & LogNormal Distributions")
axs[1].set_xlabel("Years")
axs[1].set_ylabel("Density")

# Create a single legend on the right side of the figure
h1, l1 = axs[0].get_legend_handles_labels()
h2, l2 = axs[1].get_legend_handles_labels()

axs[0].legend(h1, l1, loc="best")
axs[1].legend(h2, l2, loc="best")

plt.tight_layout()  # Adjust layout to fit legend
if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "Figure_3.png")
    plt.savefig(file_path, dpi=DPI)
plt.show()

## Cohorts in stock, and fit with total stock

In [ ]:
sns.set_theme(context="paper", style="whitegrid", palette=COLOR_PALETTE)

# Use a diverging or sequential colormap with black edges for separation
colors = get_colors("Spectral", 13)  # FIXME hardcoded;  'colorblind', 'turbo', 'viridis'?

fig, ax = plt.subplots(1, 2, figsize=(7, 3.5), dpi=150)
ax = ax.flatten()
dataset[
    (dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] != "1608-2025")
].astype({"census_year": "int"}).groupby(["census_year", "vintage"])[
    "dwellings"
].sum().unstack("vintage").plot(
    kind="bar", stacked=True, ax=ax[0], color=colors, edgecolor="white", linewidth=0.5
)  # plot

(
    census_dw[
        (census_dw["type"].isin(TARGET_TYPES + ["total"]))
        & (census_dw["vintage"] == "1608-2025")
    ].pivot(index=["census_year"], columns="type", values="dwellings")
).loc[:, "total"].plot(style="o", ax=ax[1], legend=False)
# ax[1].set_yscale('log')

# compare to 'only' totals, instead of the census_dw values
dataset[
    (dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] == "1608-2025")
].astype({"census_year": "int"}).groupby(["census_year", "vintage"])[
    "dwellings"
].sum().unstack("vintage").loc[:, ["1608-2025"]].plot(
    style=".", color="r", ax=ax[1], legend=False
)  # plot

handles, labels = ax[1].get_legend_handles_labels()
labels = ["census", "model"]
ax[1].legend(handles, labels, loc="best")

sns.move_legend(
    ax[0], loc="upper left", bbox_to_anchor=[0, -0.2], ncols=4, frameon=False
)

In [ ]:
sns.set_theme(context="paper", style="whitegrid", palette=COLOR_PALETTE)

# Use a diverging or sequential colormap with black edges for separation
colors = get_colors("Spectral", 13)  # or 'turbo', 'viridis'; 'colorblind'?

# Use gridspec for 60:40 split
fig, ax = plt.subplots(
    1, 2, figsize=(9, 4), dpi=150, gridspec_kw={"width_ratios": [3, 2]}
)
ax = ax.flatten()

# Prepare data for left plot
data_to_plot = (
    dataset[(dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] != "1608-2025")]
    .astype({"census_year": "int"})
    .groupby(["census_year", "vintage"])["dwellings"]
    .sum()
    .unstack("vintage")
)

data_to_plot.plot(
    kind="bar",
    stacked=True,
    ax=ax[0],
    color=colors,
    edgecolor="white",
    linewidth=0.3,
    width=0.8,
)

# Improve x-axis legibility
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax[0].set_xlabel("")  # Remove redundant label if present
ax[0].set_ylabel("Total dwellings", fontsize=9)

# Reduce number of x-ticks if too many (show every nth label)
n_ticks = len(data_to_plot.index)
if n_ticks > 15:
    # Show every 2nd or 3rd tick
    step = 2 if n_ticks <= 25 else 3
    for i, label in enumerate(ax[0].xaxis.get_ticklabels()):
        if i % step != 0:
            label.set_visible(False)

# Right subplot
census_dw[census_dw["vintage"] == "1608-2025"].pivot(
    index=["census_year"], columns="type", values="dwellings"
).loc[:, "total"].plot(style="o", ax=ax[1], legend=False, markersize=5)

# compare to total values from reconciled dataset
dataset[
    (dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] == "1608-2025")
].astype({"census_year": "int"}).groupby(["census_year", "vintage"])[
    "dwellings"
].sum().unstack("vintage").loc[:, ["1608-2025"]].plot(
    kind="line", color="k", ax=ax[1], legend=False, markersize=5
)

ax[1].set_xlabel("")
ax[1].set_ylabel("Total Dwellings", fontsize=9)
ax[1].tick_params(axis="x", rotation=45, labelsize=8)

handles, labels = ax[1].get_legend_handles_labels()
labels = [
    "Census data",
    "Interpolated",
]

ax[1].legend(
    handles,
    labels,
    loc="upper left",
    bbox_to_anchor=[0, -0.25],
    ncols=4,
    frameon=False,
    fontsize=8,
    title=None,
)
ax[1].set_yscale("log")

# Move legend below left plot with better spacing
sns.move_legend(
    ax[0],
    loc="upper left",
    bbox_to_anchor=[0, -0.25],
    ncols=4,
    frameon=False,
    fontsize=8,
    title=None,
)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.subplots_adjust(bottom=0.30)  # Make room for legend below

if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "Figure_5.png")
    plt.savefig(file_path, dpi=DPI)

plt.show()

# Calibration model
This is a model with exogenous stocks, were we attempt to reproduce historic inflows and outflows by varying the lifetime assumptions. The calibration process cannot produce a trivial solution - its trying to go from the integral (stocks) back to its individual components. 

In general, we can directly access the ODYM 'dsm_model', which contains: 
- t: time
- i: inflows
- o: outflows
- s: stocks
- lt: lifetime
- s_c: stock by cohort
- o_c: outflows by cohort
- name="DSM"
- pdf: probability density function
- sf: survival function
- removed_cg: removed cohorts and types

Low NRMSE values indicates the model matches historic constructions (inflows) or demolitions (outflows). The 'trust' towards historic flows is balanced through weights.

Although outflows are small (~2000 dwellings/yr on average over the known historic period 1968-1999), they have a large impact on model calibration, as outflows are the cumulative 'sum' of PDF of each individual inflows. Outflows seem like a driving factor towards longer lifetimes in the model.

Outflow data is also typically less well documented than inflow; in Canada, housing completions are confirmed by the CHMC, while demolition data comes from permit. Outflows might be underrepresented. Missed conversions, subdivisions, other demolitions (e.g, fire), etc., can also affect the actual model fit.

## Description
Concretely, the function:
1. Builds a lifetime distribution (Normal, Weibull, LogNormal2) from the passed parameters
2. Runs the stock model
3. Computes prediction errors by comparing outputs with known historic stocks (at census years) and flows (at observed years), based on census data, nh_data, and nh_demolished
4. Normalizes each error as a normalized root mean squared error (NMRSE), the RMSE divided by the mean of the series to make it unitless
<!-- FIXME: consider instead using another value to normalize, like range or max() -->
5. Combines the stock, inflow and outflow errors into a scalar, using user-defined weights (by default, [0.5, 0.25, 0.25], effectively [0.4, 0.30, 0.30] in this example).
6. Penalizes negative inflows
7. Optimizes the returned scalar, minimizing it.

In formula form:

$$
\text{NRMSE}(y,\hat{y}) =
\frac{
\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}
}{
\overline{y}
}
$$

$$
\text{Objective} =
w_{\text{stock}} \cdot \text{NRMSE}_{\text{stock}}
+
w_{\text{inflow}} \cdot \text{NRMSE}_{\text{inflow}}
+
w_{\text{outflow}} \cdot \text{NRMSE}_{\text{outflow}}
+
\underbrace{
\sum_t \max(-i_t,0)\cdot 10^3
}_{\text{negative inflow penalty}}
$$

With $(w_{\text{stock}}, w_{\text{inflow}}, w_{\text{outflow}})$ as weights (e.g.,  `CALIB_WEIGHTS = (0.4, 0.30, 0.30)`).

## On NegativeInflowCorrect
The model results are sensitive to 'NegativeInflowCorrect', in ODYM's dynamic stock model. In the calibration process, there's a penalty to prevent negative inflows, which are not possible and should not occur. However, some lifetime assumptions *can* lead to small negative inflows in the 1600s and 1700s due to numerical artifacts and/or inaccuracies resulting from the IPFN process (e.g., 1685 total stock is higher than 1688). Rather than avoiding *all* negative inflows, the penalty simply penalizes it.

When 'NegativeInflowCorrect' is set to True, then the model forces mass balance. When stock declines faster than the lifetime distribution would predict, the inflow of the given cohort is set to zero, to match the expected stock. Then, the 'extra' stock is removed proportionnally from all existing cohorts. 

The removed dwellings can be accessed through the 'dsm_model' `removed_cg`. The negative inflow penalty needs to be validated, as it can have a large effect on calibration. It can be several orders of magnitude higher than the NRMSE values (which are typically 0.1-0.5). This can lead to problems (i.e., overfitting to avoid ANY negative inflow, where it's basically a non-issue in the model), by heavily favouring shorter lifetimes to avoid these early negative inflows.

## On the RSD choice for Normal PDF
RSD=0.3 is based on similar models from the literature. However, the value is sensitive. For a given average, high values (>0.5) lead to wide distributions, which both increase early and late demolitions. Low values (<0.2) lead to very central distributions, closer to single 'pulses'.

Increasing RSD increases the estimation of flows, as more dwellings are built/demolished before or after the mean. For any given year, the stock balance is S(t) = S(t-1) + I(t) - O(t). Thus, I(t) = S(t) - S(t-1) + O(t), or $\Delta(S)$ + O(t). The outflows O are a convolution of past inflows with a survival curve (discrete pdf, or cohort-loss fraction). The lifetime parameters affects the outflows through the shape of this survival function, and I(t) then follows to meet the stock balance. As RSD (coeff. of variation) increases, the PDF flattens, and the survival curve declines more slowly. This increases the fraction of short lifetime *and* long lifetimes (more early retirements, long tail). The *before* part is quite critical here. Reducing RSD reduces the estimation of flows, as dwellings are built/demolished closer to target year. If there are fewer early demolitions, this proportionnally reduces the required future inflows to meet the desired stocks. Using PDF with high early *hazard rates* thus seems undesirable. Conversely, with small RSD, fewer outflows occur before the mean, which reduces the need for 'early' retirements.

In [ ]:
def run_dsm_scenario(
    model_time,
    total_stock,
    lifetime_dist,
    updated_tsplit,
    model_cohorts=None,  # FIXME unused
    negative_inflow_correct=False,
):
    """
    Run a single DSM scenario and return results.

    Parameters
    ----------
    model_time : np.ndarray
        Time array (years)
    total_stock : pd.Series or np.ndarray
        Total dwelling stock over time
    lifetime_dist : dict
        Lifetime distribution parameters, e.g.:
        - {'Type': 'Normal', 'Mean': 100, 'StdDev': 30}
        - {'Type': 'Weibull', 'Shape': 2.6, 'Scale': 150}
        - {'Type': 'LogNormal2', 'Mean': 4.6, 'StdDev': 0.57}
    updated_tsplit : pd.DataFrame
        Type split fractions over time
    model_cohorts : dict, optional
        Cohort definitions for grouping

    Returns
    -------
    dict with keys: 'inflows', 'outflows', 'stock_by_cohort', 'dsm_model'
    """
    n_years = len(model_time)

    # Build distribution arrays
    dist_params = {"Type": lifetime_dist["Type"]}

    if lifetime_dist["Type"] in ["Normal", "LogNormal2"]:
        dist_params["Mean"] = np.full((n_years, 1), lifetime_dist["Mean"])
        dist_params["StdDev"] = np.full((n_years, 1), lifetime_dist["StdDev"])
    elif lifetime_dist["Type"] == "Weibull":
        dist_params["Shape"] = np.full((n_years, 1), lifetime_dist["Shape"])
        dist_params["Scale"] = np.full((n_years, 1), lifetime_dist["Scale"])

    # Create and run DSM
    stock_array = (
        total_stock.to_numpy()
        if hasattr(total_stock, "to_numpy")
        else total_stock
    )

    dsm_model = dsm.DynamicStockModel(
        t=model_time,
        s=stock_array,
        lt=dist_params,
    )
    dsm_model.dimension_check()
    S_C, O_C, I = (
        dsm_model.compute_stock_driven_model(
            NegativeInflowCorrect=negative_inflow_correct
        )
    )
    dsm_model.compute_outflow_total()
    # TODO add dsm_model.remove_cg if negative_inflow_correct is true, to check what was removed (and when)

    # Calculate type-weighted flows
    inflows = pd.DataFrame(
        np.einsum("t,tj->t", dsm_model.i.reshape(-1), updated_tsplit.to_numpy()),
        index=model_time,
        columns=["inflow"],
    )
    outflows = pd.DataFrame(
        np.einsum("t,tj->t", dsm_model.o.reshape(-1), updated_tsplit.to_numpy()),
        index=model_time,
        columns=["outflow"],
    )

    return {
        "inflows": inflows,
        "outflows": outflows,
        "S_C": S_C,
        "O_C": O_C,
        "dsm_model": dsm_model,
    }


def run_sensitivity_analysis(
    model_time,
    total_stock,
    updated_tsplit,
    scenarios,
):
    """
    Run multiple DSM scenarios for sensitivity analysis.

    Parameters
    ----------
    scenarios : list of dict
        Each dict has 'label' and 'dist' keys, e.g.:
        [
            {'label': 'Normal_50', 'dist': {'Type': 'Normal', 'Mean': 50, 'StdDev': 15}},
            {'label': 'Weibull_Deetman', 'dist': {'Type': 'Weibull', 'Shape': 1.97, 'Scale': 57.53}},
        ]

    Returns
    -------
    dict : {label: results_dict}
    """
    results = {}
    for scenario in scenarios:
        results[scenario["label"]] = run_dsm_scenario(
            model_time, total_stock, scenario["dist"], updated_tsplit
        )
    return results


def plot_sensitivity_results(
    results,
    nh_data=None,
    nh_demolished=None,
    inflow_xlim=(1948, 2023),
    outflow_xlim=(1965, 2005),
    outflow_ylim=(0, 25000),
    figsize=(7, 7),
    sharex=True,
    cmap="Spectral",
):
    """Plot inflows/outflows from sensitivity analysis results."""
    fig, ax = plt.subplots(2, 1, figsize=figsize)

    labels = list(results.keys())
    n_results = len(results)

    # Generate colors from cmap if not provided.
    colors = get_colors(cmap, 13)  # or 'turbo', 'viridis'; 'colorblind'?

    for i, (label, res) in enumerate(results.items()):
        res["inflows"].plot(ax=ax[0], legend=False, color=colors[i], label=label)
        res["outflows"].plot(ax=ax[1], legend=False, color=colors[i], label=label)

    # Historic data
    if nh_data:
        ax[0].plot(nh_data.keys(), nh_data.values(), "k.", label="Historic")
    if nh_demolished is not None:
        ax[1].plot(nh_demolished["REF_DATE"], nh_demolished["VALUE"], "k.")

    ax[0].set(xlabel="Year", ylabel="Dwelling completions", xlim=inflow_xlim)
    ax[1].set(
        xlabel="Year",
        ylabel="Dwelling demolitions",
        xlim=inflow_xlim if sharex else outflow_xlim,  # NOTE: if sharex needed
        ylim=outflow_ylim,
    )

    fig.legend(
        ax[0].get_legend_handles_labels()[0],
        labels,
        loc="upper center",
        bbox_to_anchor=[0.5, 0],
        frameon=False,
        ncols=min(len(labels), 4),
    )

    # ---- Add vertical lines for historic phases ----
    phase_lines = {
        "A": 1976,
        "B": 1982,
        "C": 1989,
        "D": 1998,
        "E": 2008,
        "F": 2019,
        "G": 2022,
    }  # https://media.apchq.com/download/a620181f9084403544a6227f8852ad0716843767.pdf

    recessions = [
        (1974, 1975),
        (1981, 1982),
        (1990, 1992),
        (2008, 2009),
    ]  # https://media.apchq.com/download/a620181f9084403544a6227f8852ad0716843767.pdf

    previous_year = min(inflow_xlim)
    for label, year in phase_lines.items():
        for axi in ax:  # apply to both subplots
            axi.axvline(
                x=year,
                linestyle="--",
                color="orange",
                linewidth=1.2,
                alpha=0.8,
            )

        # Add label above the first panel
        ax[0].text(
            np.mean([year, previous_year]),
            ax[0].get_ylim()[1] * 0.99,  # relative to y-max
            label,
            ha="center",
            va="bottom",
            fontsize=11,
            color="white",
            bbox=dict(boxstyle="circle,pad=0.25", fc="orange", ec="none"),
        )
        previous_year = year

    # ---- Gray recession bands ----
    for start, end in recessions:
        for axi in ax:
            axi.axvspan(
                start,
                end,
                color="gray",
                alpha=0.25,
                linewidth=0,  # no border
                zorder=0,  # behind data
            )

    plt.tight_layout()
    return fig, ax

In [ ]:
# Run analysis
results = run_sensitivity_analysis(
    MODEL_TIME, total_stock, updated_tsplit, BASE_SCENARIOS
)

# Plot
# Use a diverging or sequential colormap with black edges for separation
fig, ax = plot_sensitivity_results(results, nh_data, nh_demolished, cmap='colorblind') # FIXME hardcoded
if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, 'Figure_6_v1.png') # early test
    plt.savefig(file_path, dpi=DPI)
plt.show()

Here, we attempt to reproduce historic inflows and outflows using different lifetime distributions:
- Deetman et al. 2019 assumptions (weibull)
- Ianchenko et al. 2020 assumptions (weibull, lognormal, gamma)
- Normal distributions of 50-200 years (COV=30%)

The values in Deetman et al. (2019) seem low; an average lifetime of 51 years for buildings in Canada is much smaller than the values in Ianchenko(2020), and also smaller than the non-residential values listed in O'Connor (2004). 

Here, we see that using an average lifetime of 100+ years provides better fit, especially for outfows; so does using the lognormal and weibull distributions from Ianchenko.

In [ ]:
# NOTE this uses the compute_tock_driven_model (no typesplit),
mask = results["Normal ($\\mu$=100)"]['dsm_model'].removed_cg > 0

results["Normal ($\\mu$=100)"]['dsm_model'].removed_cg[mask]

Adapted from the [Bulletin de l'habitation (2023) de l'APCHQ](https://media.apchq.com/download/a620181f9084403544a6227f8852ad0716843767.pdf), itself containing data from CHMC. The data presented here are *housing starts*, not  *housing completions*. 
\citep(apchq2023)

| Années        | Phase | Mises en chantier (moyennes annuelles) | Périodes                         | Faits saillants                                                                 |
|---------------|:-----:|-----------------------------------------|----------------------------------|----------------------------------------------------------------------------------|
| 1960 à 1976   | A     | 46 284                                  | Les années boomers, 1re partie   | Les premiers baby-boomers se ruent sur le marché locatif                         |
| 1977 à 1982   | B     | 37 551                                  | Le krach                         | Choc pétrolier, inflation, taux d’intérêt exorbitants et récession              |
| 1983 à 1989   | C     | 53 128                                  | Les années boomers, 2e partie    | Les baby-boomers accèdent massivement à la propriété                             |
| 1990 à 1998   | D     | 32 584                                  | La grande noirceur               | Récession et lutte aux déficits budgétaires : les consommateurs étouffés         |
| 1999 à 2008   | E     | 42 455                                  | La Renaissance                   | La conjoncture s’améliore grandement, le prix des propriétés double             |
| 2009 à 2019   | F     | 44 117                                  | L’évolution tranquille           | La baisse des taux d’intérêt se poursuit, le marché immobilier est résilient    |
| 2020 à 2022   | G     | 59 661                                  | Les Années folles                | La pandémie de COVID-19 crée une frénésie                                       |


## TEST for different typesplit treatment

In [ ]:
AVG_LIFETIME = [50, 75, 100, 150, 200]
MODEL_CAP = LAST_MDL_YR
NEG_INFLOW_CORRECT = True

Ng = updated_tsplit.shape[1]  # Number of dwelling types

# No historic stock - start from year 0
SwitchTime = 0

fig, ax = plt.subplots(2, 1, figsize=(7, 7))
ax = ax.flatten()

colors = get_colors('tab10',  len(AVG_LIFETIME))

for i, lifetime in enumerate(AVG_LIFETIME):
    beta_lt = np.full((Nt,), lifetime)
    beta_std = beta_lt * RSD

    # Build SFArrayCombined[t, c, g] - same lifetime for all types
    SFArrayCombined = np.zeros((Nt, Nt, Ng))
    for c in range(Nt):
        if beta_lt[c] != 0:
            sf_values = stats.norm.sf(
                np.arange(0, Nt - c), loc=beta_lt[c], scale=beta_std[c]
            )
            for g in range(Ng):
                SFArrayCombined[c:, c, g] = sf_values

    # Zero initial stock - shape (Nt, Ng)
    InitialStock = np.zeros((Nt, Ng))

    # TypeSplit[t, g] - must sum to 1 per year for future years
    TypeSplit = updated_tsplit.to_numpy()  # Shape (Nt, Ng)

    # Create DSM - note: lt is still needed for the method to run
    qc_dwlg_dsm = dsm.DynamicStockModel(
        t=MODEL_TIME,
        s=total_stock.to_numpy(),
        lt={"Type": "Normal", "Mean": beta_lt, "StdDev": beta_std},
    )

    # Run the typesplit model
    s_cg, o_cg, i_g, NIC_Flags = (
        qc_dwlg_dsm.compute_stock_driven_model_initialstock_typesplit_negativeinflowcorrect(
            SwitchTime=SwitchTime,
            InitialStock=InitialStock,
            SFArrayCombined=SFArrayCombined,
            TypeSplit=TypeSplit,
            NegativeInflowCorrect=NEG_INFLOW_CORRECT,
            NegativeInflowStyle="Default",
        )
    )

    # Total inflow (summed across types)
    total_inflow = i_g.sum(axis=1)

    # Total outflow (summed across cohorts and types)
    total_outflow = o_cg.sum(axis=(1, 2))

    # Plot
    pd.DataFrame(total_inflow, index=MODEL_TIME).loc[:MODEL_CAP, :].plot(
        ax=ax[0], color=colors[i], legend=True
    )
    pd.DataFrame(total_outflow, index=MODEL_TIME).loc[:MODEL_CAP, :].plot(
        ax=ax[1], color=colors[i], legend=True
    )
    # FIXME use MODEL_CAP for xlim instead?

# Rest of your plotting code...# Add legend with lifetime labels
ax[0].legend(
    [f"Lifetime {lt}y" for lt in AVG_LIFETIME],
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
)
ax[1].legend(
    [f"Lifetime {lt}y" for lt in AVG_LIFETIME],
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
)

# Plot historic data if available
if "nh_data" in globals() and nh_data is not None:
    ax[0].plot(list(nh_data.keys()), list(nh_data.values()), "k.", label="Historic")

if "nh_demolished" in globals() and nh_demolished is not None:
    ax[1].plot(nh_demolished["REF_DATE"], nh_demolished["VALUE"], "k.")

ax[0].set_xlabel("Year")
ax[0].set_ylabel("Dwelling completions (inflows)")
# ax[0].set_xlim([1948, 2023])

ax[1].set_xlabel("Year")
ax[1].set_ylabel("Dwelling demolitions (outflows)")
# ax[1].set_xlim([1965, 2005])
ax[1].set_ylim([0, 25000])

plt.tight_layout()

if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "Figure_X.png")
    plt.savefig(file_path, dpi=DPI)

plt.show()

In [ ]:
def run_dsm_scenario_typesplit(
    model_time,
    total_stock,
    lifetime_dist,
    updated_tsplit,
    negative_inflow_correct=False,
    negative_inflow_style="Default",
):
    """
    Run a single DSM scenario using typesplit method and return results.

    Parameters
    ----------
    model_time : np.ndarray
        Time array (years)
    total_stock : pd.Series or np.ndarray
        Total dwelling stock over time
    lifetime_dist : dict
        Lifetime distribution parameters
    updated_tsplit : pd.DataFrame
        Type split fractions over time (Nt, Ng)

    Returns
    -------
    dict with keys: 'inflows', 'outflows', 'inflows_by_type', 'stock_by_cohort_type', 'dsm_model'
    """
    Nt = len(model_time)  # FIXME same as 'global' value?
    Ng = updated_tsplit.shape[1]

    # No historic stock - start from year 0
    SwitchTime = 0

    # Build lifetime arrays
    if lifetime_dist["Type"] == "Normal":
        try:
            beta_lt = np.full((Nt,), lifetime_dist["Mean"])
            beta_std = np.full((Nt,), lifetime_dist["StdDev"])
        except ValueError as e:
            print(
                f"Encountered {e}. Attempting a reshape."
            )
            beta_lt = np.full((Nt,), lifetime_dist["Mean"].reshape(-1))
            beta_std = np.full((Nt,), lifetime_dist["StdDev"].reshape(-1))

        # Build SFArrayCombined[t, c, g] using normal survival function
        SFArrayCombined = np.zeros((Nt, Nt, Ng))
        for c in range(Nt):
            sf_values = stats.norm.sf(
                np.arange(0, Nt - c), loc=beta_lt[c], scale=beta_std[c]
            )
            for g in range(Ng):
                SFArrayCombined[c:, c, g] = sf_values

    elif lifetime_dist["Type"] == "Weibull":
        shape = lifetime_dist["Shape"]
        scale = lifetime_dist["Scale"]

        # Build SFArrayCombined using Weibull survival function
        SFArrayCombined = np.zeros((Nt, Nt, Ng))
        for c in range(Nt):
            sf_values = stats.weibull_min.sf(np.arange(0, Nt - c), c=shape, scale=scale)
            for g in range(Ng):
                SFArrayCombined[c:, c, g] = sf_values

        # For DSM initialization (needed even if not directly used)
        beta_lt = np.full((Nt,), scale)
        beta_std = np.full((Nt,), scale * 0.3)  # Placeholder  # FIXME

    elif lifetime_dist["Type"] == "LogNormal2":
        mu_ln = lifetime_dist["Mean"]
        sigma_ln = lifetime_dist["StdDev"]

        # Build SFArrayCombined using lognormal survival function
        SFArrayCombined = np.zeros((Nt, Nt, Ng))
        for c in range(Nt):
            sf_values = stats.lognorm.sf(
                np.arange(0, Nt - c), s=sigma_ln, scale=np.exp(mu_ln)
            )
            for g in range(Ng):
                SFArrayCombined[c:, c, g] = sf_values

        # For DSM initialization
        beta_lt = np.full((Nt,), np.exp(mu_ln))
        beta_std = np.full((Nt,), np.exp(mu_ln) * 0.3)  # Placeholder # FIXME

    # Zero initial stock - shape (Nt, Ng)
    InitialStock = np.zeros((Nt, Ng))

    # TypeSplit[t, g] - must sum to 1 per year
    TypeSplit = updated_tsplit.to_numpy()

    # Total stock time series
    stock_array = (
        total_stock.to_numpy()
        if hasattr(total_stock, "to_numpy")
        else total_stock
    )

    # Create DSM
    dsm_model = dsm.DynamicStockModel(
        t=model_time,
        s=stock_array,
        lt={"Type": "Normal", "Mean": beta_lt, "StdDev": beta_std},
    )

    # Run the typesplit model
    s_cg, o_cg, i_g, NIC_Flags = (
        dsm_model.compute_stock_driven_model_initialstock_typesplit_negativeinflowcorrect(
            SwitchTime=SwitchTime,
            InitialStock=InitialStock,
            SFArrayCombined=SFArrayCombined,
            TypeSplit=TypeSplit,
            NegativeInflowCorrect=negative_inflow_correct,
            NegativeInflowStyle=negative_inflow_style,  # NOTE: usually Default
        )
    )

    # Total inflow (summed across types)
    total_inflow = i_g.sum(axis=1)

    # Total outflow (summed across cohorts and types)
    total_outflow = o_cg.sum(axis=(1, 2))

    # Create DataFrames
    inflows = pd.DataFrame(total_inflow, index=model_time, columns=["inflow"])
    outflows = pd.DataFrame(total_outflow, index=model_time, columns=["outflow"])
    inflows_by_type = pd.DataFrame(
        i_g, index=model_time, columns=updated_tsplit.columns
    )

    # TODO add dsm_model.remove_cg if negative_inflow_correct is true, to check what was removed (and when)

    return {
        "inflows": inflows,
        "outflows": outflows,
        "inflows_by_type": inflows_by_type,
        "s_cg": s_cg,
        "o_cg": o_cg,
        "i_g": i_g,
        "NIC_Flags": NIC_Flags,
        "dsm_model": dsm_model,
    }


def run_sensitivity_analysis_typesplit(
    model_time,
    total_stock,
    updated_tsplit,
    scenarios,
    negative_inflow_correct=False,
    negative_inflow_style="Default",
):  # FIXME order here is different than in run_dsm_scenario_typesplit, it's confusing
    """
    Run multiple DSM scenarios using typesplit method.
    """
    results = {}
    for scenario in scenarios:
        print(f"Running scenario: {scenario['label']}")
        results[scenario["label"]] = run_dsm_scenario_typesplit(
            model_time,
            total_stock,
            scenario["dist"],
            updated_tsplit,
            negative_inflow_correct=negative_inflow_correct,
            negative_inflow_style=negative_inflow_style,
        )
    return results


def plot_sensitivity_results_typesplit(
    results,
    nh_data=None,
    nh_demolished=None,
    inflow_xlim=(1948, 2023),
    outflow_xlim=(1965, 2005),
    outflow_ylim=(0, 25000),
    figsize=(7, 7),
    share_ax=True,
    # FIXME add color control here (cmap)
):
    """Plot inflows/outflows from sensitivity analysis results."""
    fig, ax = plt.subplots(2, 1, figsize=figsize)

    labels = list(results.keys())
    for label, res in results.items():
        res["inflows"].plot(ax=ax[0], legend=False)
        res["outflows"].plot(ax=ax[1], legend=False)

    # Historic data
    if nh_data:
        ax[0].plot(nh_data.keys(), nh_data.values(), "k.", label="Historic")
    if nh_demolished is not None:
        ax[1].plot(nh_demolished["REF_DATE"], nh_demolished["VALUE"], "k.")

    ax[0].set(xlabel="Year", ylabel="Dwelling completions", xlim=inflow_xlim)
    ax[1].set(
        xlabel="Year",
        ylabel="Dwelling demolitions",
        xlim=inflow_xlim if share_ax else outflow_xlim,
        ylim=outflow_ylim,
    )

    fig.legend(
        ax[0].get_legend_handles_labels()[0],
        labels,
        loc="upper center",
        bbox_to_anchor=[0.5, 0],
        frameon=False,
        ncols=min(len(labels), 4),
    )
    # ---- Add vertical lines for historic phases ----
    phase_lines = {
        "A": 1976,
        "B": 1982,
        "C": 1989,
        "D": 1998,
        "E": 2008,
        "F": 2019,
        "G": 2022,
    }  # https://media.apchq.com/download/a620181f9084403544a6227f8852ad0716843767.pdf

    recessions = [
        (1974, 1975),
        (1981, 1982),
        (1990, 1992),
        (2008, 2009),
    ]  # https://media.apchq.com/download/a620181f9084403544a6227f8852ad0716843767.pdf

    previous_year = min(inflow_xlim)
    for label, year in phase_lines.items():
        for axi in ax:  # apply to both subplots
            axi.axvline(
                x=year,
                linestyle="--",
                color="orange",
                linewidth=1.2,
                alpha=0.8,
            )

        # Add label above the first panel
        ax[0].text(
            np.mean([year, previous_year]),
            ax[0].get_ylim()[1] * 0.99,  # relative to y-max
            label,
            ha="center",
            va="bottom",
            fontsize=11,
            color="white",
            bbox=dict(boxstyle="circle,pad=0.25", fc="orange", ec="none"),
        )
        previous_year = year

    # ---- Gray recession bands ----
    for start, end in recessions:
        for axi in ax:
            axi.axvspan(
                start,
                end,
                color="gray",
                alpha=0.25,
                linewidth=0,  # no border
                zorder=0,  # behind data
            )

    plt.tight_layout()
    return fig, ax

In [ ]:
# Run analysis with typesplit method
results_typesplit = run_sensitivity_analysis_typesplit(
    MODEL_TIME,
    total_stock,
    updated_tsplit,
    BASE_SCENARIOS,
    negative_inflow_correct=NEG_INFLOW_CORRECT,
    negative_inflow_style="Default", # FIXME hardcoded
)

# Plot
fig, ax = plot_sensitivity_results_typesplit(
    results_typesplit,
    nh_data,
    nh_demolished,
)

if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "Figure_6_v2.png") # w/ typesplit
    plt.savefig(file_path, dpi=DPI)

plt.show()

# NOTE/FIXME: change style by type of distribution! normal, lognormal, weibull should be more easily distinguishable (e.g., not only colour)

# FIXME: record ALL YEARS W NEGATIVE FLOWS : in dynamicstockmodel, change the warning behaviour to record and output ALL years w negative inflows, or perhaps check them here using the model? i by cohort and type?

Due to only having access to stock typesplit, in this approach, stock by type is computed, then inflows derived to meet type-specific targets. 

For the current analysis where all types share the same lifetime, results are equivalent to the approach of Roca-Puigros et al., where typesplit doesn't influence the stock balance - it's purely a disaggregation of already-computed flows. However, this method enables future extensions like type-cohort specific lifetimes. this approach is more general and flexible for heterogeneous building stocks.

In [ ]:
# Check if there are any remaining negative inflows
scenario_key = "Normal ($\\mu$=150)"
mask = results_typesplit[scenario_key]["dsm_model"].removed_cg > 0  # Shape: (Nt, Nc, Ng)
results_typesplit[scenario_key]["dsm_model"].removed_cg[mask]

In [ ]:
fig, ax = plt.subplots()
# interpolated data
# Total stock in census
# tmp['vintage'] is already based on cohorts_2021, and includes 'total'

df_c = pd.DataFrame(
    dataset[(dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] == "1608-2025")]
    .groupby("census_year")["dwellings"]
    .sum()
)

# Decadal stock change
# from the dataset itself, and a simple linear interpolation FIXME hardcoded
df_c.reindex(list(range(1931, 2021))).interpolate(method="linear").diff().plot(
    style="o", ax=ax
)

# from my own model
total_stock.loc[1931:2021].diff().rename("ipfn").plot(
    style=".--", ax=ax
)  # normal, same as df_c


# last 'inflow' calculated by my model
tt = results_typesplit[scenario_key]['inflows'].loc[MODEL_TIME] # retrieve inflows
# tt.index = MODEL_TIME
tt.loc[1968:1999].plot(style=":", ax=ax)

# calculated stock change from completions and demolitions
df_stock_change = df_b.reindex_like(df_a) - df_a
df_stock_change.plot(style=".", ax=ax)

ax.set_title("Stock change, 1968-1999")
ax.set_xlim([1960, 2005])

In [ ]:
# Select a scenario to visualize (e.g., Normal_100)
scenario_key = "Normal ($\\mu$=150)"
s_cg = results_typesplit[scenario_key]["s_cg"]  # Shape: (Nt, Nc, Ng)

# Create aggregated stock by cohort bin and type
stock_by_cohort_type = {}

for start_yr, end_yr, label in cohort_bins:
    c_start = max(0, start_yr - MODEL_TIME[0])
    c_end = min(Nt, end_yr - MODEL_TIME[0] + 1)
    stock_cohort = s_cg[:, c_start:c_end, :].sum(axis=1)

    for g, type_name in enumerate(TARGET_TYPES):
        key = (label, type_name)
        stock_by_cohort_type[key] = stock_cohort[:, g]

# Convert to DataFrame for plotting
stock_df = pd.DataFrame(stock_by_cohort_type, index=MODEL_TIME)
stock_df.columns = pd.MultiIndex.from_tuples(stock_df.columns, names=["cohort", "type"])

# Reshape for seaborn
stock_long = stock_df.stack(["cohort", "type"]).reset_index()
stock_long.columns = ["year", "cohort", "type", "dwellings"]

# Filter to relevant time period
stock_long = stock_long[
    (stock_long["year"] >= TIME_WINDOW[0]) & (stock_long["year"] <= TIME_WINDOW[1])
]

# Create the figure
sns.set_theme(context="paper", style="whitegrid", palette=COLOR_PALETTE)

fig, axs = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# Get color palette for cohorts
cohort_order = [c[2] for c in cohort_bins]
cohort_palette = sns.color_palette(COLOR_PALETTE, len(cohort_bins))
cohort_colors = {label: cohort_palette[i] for i, label in enumerate(cohort_order)}

# Linestyles for types
type_linestyles = {
    TARGET_TYPES[i]: ["-", "--", "-.", ":"][i % 4] for i in range(len(TARGET_TYPES))
}

# Panel 1: Line plot - Stock evolution by cohort, with hue as cohort and linestyle as type
ax1 = axs[0]
for cohort_label in cohort_order:
    cohort_data = stock_long[stock_long["cohort"] == cohort_label]
    for type_name in TARGET_TYPES:
        type_data = cohort_data[cohort_data["type"] == type_name]
        ax1.plot(
            type_data["year"],
            type_data["dwellings"],
            color=cohort_colors[cohort_label],
            linestyle=type_linestyles[type_name],
            linewidth=1.5,
            alpha=0.8,
        )

ax1.set_xlabel("Year")
ax1.set_ylabel("Dwelling stock")
ax1.set_title(f"Stock evolution by cohort and type ({scenario_key})")
ax1.set_xlim(TIME_WINDOW[0], TIME_WINDOW[1])

# Panel 2: Stacked area plot - Total stock by cohort
ax2 = axs[1]
stock_by_cohort = stock_df.groupby(level="cohort", axis=1).sum()
stock_by_cohort = stock_by_cohort.loc[TIME_WINDOW[0]:TIME_WINDOW[1]]
stock_by_cohort = stock_by_cohort[cohort_order]

stock_by_cohort.plot(
    kind="area", stacked=True, ax=ax2, alpha=0.8, color=cohort_palette, legend=False
)
ax2.set_xlabel("Year")
ax2.set_ylabel("Dwelling stock")
ax2.set_title("Stock by cohort (all types)")

# Create combined legend handles
# Cohort legend (colors)
cohort_handles = [
    plt.Line2D([0], [0], color=cohort_colors[label], linewidth=2, label=label)
    for label in cohort_order
]

# Type legend (linestyles)
type_handles = [
    plt.Line2D(
        [0],
        [0],
        color="black",
        linestyle=type_linestyles[t],
        linewidth=2,
        label=t.replace("_", " ").title(),
    )
    for t in TARGET_TYPES
]

# Add figure legend at the bottom
fig.legend(
    handles=cohort_handles + type_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=5,
    frameon=False,
    title="Cohort (color) / Type (linestyle)",
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)  # Make room for legend
if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "s_cg.png")
    plt.savefig(file_path, dpi=DPI)

plt.show()

Note for interpretation: this figure can be somewhat hard to read, it sometimes seems like the lines of a same colour should 'touch', which is not the case. It only depicts the evolution of stocks, by type (line style) and cohort (hue).

# Try to find "better" models (lifetime calibration)

In [ ]:
def calculate_model_fit_metrics(observed_stock, predicted_stock):
    """
    Calculate multiple error metrics for model calibration.

    Args:
        observed_stock: Census stock values (array-like)
        predicted_stock: Model predicted stock values (array-like)

    Returns:
        dict: Dictionary of error metrics
    """
    obs = np.array(observed_stock)
    pred = np.array(predicted_stock)

    # Filter out NaN values
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    obs, pred = obs[mask], pred[mask]

    metrics = {}

    # 1. Root Mean Square Error (RMSE) - penalizes large errors
    metrics["rmse"] = np.sqrt(np.mean((obs - pred) ** 2))

    # 2. Normalized RMSE (as % of mean observed value)
    metrics["nrmse"] = metrics["rmse"] / np.mean(obs) * 100

    # 3. Mean Absolute Error (MAE) - more robust to outliers
    metrics["mae"] = np.mean(np.abs(obs - pred))

    # 4. Mean Absolute Percentage Error (MAPE)
    metrics["mape"] = np.mean(np.abs((obs - pred) / obs)) * 100

    # 5. R-squared (coefficient of determination)
    ss_res = np.sum((obs - pred) ** 2)
    ss_tot = np.sum((obs - np.mean(obs)) ** 2)
    metrics["r_squared"] = 1 - (ss_res / ss_tot)

    return metrics


def objective_function(
    params,
    model_time,
    target_stock,
    census_years,
    census_values,
    dist_type="Normal",
    negative_inflow_correct=False,
):
    """
    Objective function for lifetime calibration.

    Args:
        params: [mean, std] or [shape, scale] depending on distribution
        model_time: Time array
        target_stock: Target stock array (total dwellings)
        census_years: Years with census observations
        census_values: Observed census stock values
        dist_type: 'Normal', 'Weibull', or 'LogNormal2'
    """
    if dist_type == "Normal":
        mean_lt, std_lt = params
        lt_dict = {
            "Type": "Normal",
            "Mean": np.full((len(model_time), 1), mean_lt),
            "StdDev": np.full((len(model_time), 1), std_lt),
        }
    elif dist_type == "Weibull":
        shape, scale = params
        lt_dict = {
            "Type": "Weibull",
            "Shape": np.full((len(model_time), 1), shape),
            "Scale": np.full((len(model_time), 1), scale),
        }
    elif dist_type == "LogNormal2":
        mu_ln, sigma_ln = params
        lt_dict = {
            "Type": "LogNormal2",
            "Mean": np.full((len(model_time), 1), mu_ln),
            "StdDev": np.full((len(model_time), 1), sigma_ln),
        }

    # Run DSM
    model = dsm.DynamicStockModel(t=model_time, s=target_stock, lt=lt_dict)

    try:
        S_C, O_C, I = model.compute_stock_driven_model(
            NegativeInflowCorrect=negative_inflow_correct
        )
        model.compute_outflow_total()

        # TODO add easier access to dsm_model.remove_cg if negative_inflow_correct is true, to check what was removed (and when)

        # Extract predicted values at census years
        year_to_idx = {yr: i for i, yr in enumerate(model_time)}
        predicted = np.array([S_C[year_to_idx[yr], :].sum() for yr in census_years])

        # Calculate RMSE (or other metric)
        rmse = np.sqrt(np.mean((census_values - predicted) ** 2))

        # Penalize negative inflows
        neg_inflow_penalty = (
            np.sum(np.maximum(-model.i, 0)) * 1e3
        )  # TODO check sensitivity: 1e3, 1e6..?

        return rmse + neg_inflow_penalty

    except Exception as e:
        return 1e12  # Return large value if model fails

In [ ]:
def multi_objective_calibration(
    params,
    model_time,
    target_stock,
    census_years,
    census_values,
    inflow_years,
    inflow_values,
    outflow_years,
    outflow_values,
    dist_type="Normal",
    weights=(0.5, 0.25, 0.25),
    negative_inflow_correct=False,
):
    """
    Multi-objective calibration using stock, inflows, and outflows.
    """
    w_stock, w_inflow, w_outflow = weights

    # Build lifetime dict
    if dist_type == "Normal":
        mean_lt, std_lt = params
        lt_dict = {
            "Type": "Normal",
            "Mean": np.full((len(model_time), 1), mean_lt),
            "StdDev": np.full((len(model_time), 1), std_lt),
        }
    elif dist_type == "Weibull":
        shape, scale = params
        lt_dict = {
            "Type": "Weibull",
            "Shape": np.full((len(model_time), 1), shape),
            "Scale": np.full((len(model_time), 1), scale),
        }
    elif dist_type == "LogNormal2":
        mu_ln, sigma_ln = params
        lt_dict = {
            "Type": "LogNormal2",
            "Mean": np.full((len(model_time), 1), mu_ln),
            "StdDev": np.full((len(model_time), 1), sigma_ln),
        }

    model = dsm.DynamicStockModel(t=model_time, s=target_stock, lt=lt_dict)

    try:
        S_C, O_C, I = (
            model.compute_stock_driven_model(
                NegativeInflowCorrect=negative_inflow_correct
            )
        )
        model.compute_outflow_total()

        # TODO add dsm_model.remove_cg if negative_inflow_correct is true, to check what was removed (and when)

        year_to_idx = {yr: i for i, yr in enumerate(model_time)}

        # TODO Stock error (NRMSE)  will be near zero for any parameters in a stock-driven model (except due to, e.g., negative-inflow corrections, or IPFN artifacts). it's mostly a feasibility indicator, so we can drop it from the calibration weights
        pred_stock = np.array([S_C[year_to_idx[yr], :].sum() for yr in census_years])
        stock_nrmse = np.sqrt(np.mean((census_values - pred_stock) ** 2)) / np.mean(
            census_values
        )

        # Inflow error (NRMSE)
        pred_inflow = np.array([model.i[year_to_idx[yr]] for yr in inflow_years])
        inflow_nrmse = np.sqrt(np.mean((inflow_values - pred_inflow) ** 2)) / np.mean(
            inflow_values
        )

        # Outflow error (NRMSE)
        pred_outflow = np.array([model.o[year_to_idx[yr]] for yr in outflow_years])
        outflow_nrmse = np.sqrt(
            np.mean((outflow_values - pred_outflow) ** 2)
        ) / np.mean(outflow_values)

        # Combined objective
        total_error = (
            w_stock * stock_nrmse + w_inflow * inflow_nrmse + w_outflow * outflow_nrmse
        )

        # Penalize negative inflows
        neg_inflow_penalty = (
            np.sum(np.maximum(-model.i, 0)) * 1e3
        )  # FIXME check sensitivity

        return total_error + neg_inflow_penalty

    except (KeyError, IndexError) as e:
        raise
    except Exception as e:
        print(f"Numerical error: {e}")
        return 1e12


def run_multi_objective_calibration(
    model_time,
    target_stock,
    census_data,
    inflow_data,
    outflow_data,
    dist_type="Normal",
    weights=(0.5, 0.25, 0.25),
    negative_inflow_correct=False,
):
    """
    Run multi-objective calibration for a given distribution type.
    """
    # Extract data
    census_years = census_data.index.to_numpy()
    census_values = census_data.to_numpy()

    inflow_years = inflow_data.index.to_numpy()
    inflow_values = inflow_data.to_numpy().flatten()

    outflow_years = outflow_data.index.to_numpy()
    outflow_values = outflow_data.to_numpy().flatten()

    # FIXME not DRY, use a single, common 'Bounds' definition
    if dist_type == "Normal":
        bounds = [(30, 300), (5, 100)]
    elif dist_type == "Weibull":
        bounds = [(1.0, 5.0), (30, 300)]
    elif dist_type == "LogNormal2":
        bounds = [(3.0, 6.0), (0.1, 1.5)]

    result = differential_evolution(
        multi_objective_calibration,
        bounds=bounds,
        args=(
            model_time,
            target_stock,
            census_years,
            census_values,
            inflow_years,
            inflow_values,
            outflow_years,
            outflow_values,
            dist_type,
            weights,
            negative_inflow_correct,
        ),
        seed=rng,
        maxiter=300,
        polish=True,
        disp=True,
    )

    return result

## Generate calibrated PDFs

In [ ]:
# Census totals
census_totals = dataset[
    (dataset["type"] == "total") & (dataset["vintage"] == "1608-2025")
].set_index("census_year")["dwellings"]

# Inflow data (dwelling completions)
inflow_data = pd.DataFrame.from_dict(
    nh_data, orient="index", columns=["completions"]
)

# Outflow data (demolitions)
outflow_data = nh_demolished.set_index("REF_DATE")["VALUE"]

TIME_SLICE = MODEL_TIME[0:-44]  # 1607-2022

# This calibration is quite long, only run it if necessary
run_this = True
if run_this:
    # === USAGE ===
    # Prepare data
    target_stock = (
        total_stock
        .loc[TIME_SLICE]  # to remove flat tail
        .to_numpy()
    )

    # Run multi-objective calibration
    print("=" * 60)
    print("Multi-objective calibration")
    print("=" * 60)

    multi_results = {}
    for dist_type in ["Normal", "Weibull", "LogNormal2"]:
        print(f"\nCalibrating {dist_type}...")
        multi_results[dist_type] = run_multi_objective_calibration(
            TIME_SLICE,  # , MODEL_TIME,
            target_stock,
            census_totals,
            inflow_data,
            outflow_data,
            dist_type,
            weights= CALIB_WEIGHTS, # e.g., (.1,0.6,0.3) or (0.4, 0.3, 0.3)
            negative_inflow_correct=True,  # NEG_INFLOW_CORRECT,
        )
        print(f"Best params: {multi_results[dist_type].x}")
        print(f"Best objective: {multi_results[dist_type].fun:.6f}")


In [ ]:
def plot_distributions(res, x_max=100):
    # Extract parameters
    mean, std_dev = res["Normal"].x
    shape_w, scale_w = res["Weibull"].x
    mu_ln, sigma_ln = res["LogNormal2"].x

    x = np.linspace(0, x_max, 1000)

    # Calculate PDFs
    y_normal = stats.norm.pdf(x, mean, std_dev)
    y_weibull = stats.weibull_min.pdf(x, shape_w, scale=scale_w)
    y_lognorm = stats.lognorm.pdf(x, sigma_ln, scale=np.exp(mu_ln))

    # Plot
    plt.figure(figsize=(12, 6))
    plt.plot(
        x,
        y_normal,
        label=f"Normal (μ={mean:.2f}, σ={std_dev:.2f})",
        color="royalblue",
        linewidth=2,
    )
    plt.plot(
        x,
        y_weibull,
        label=f"Weibull (c={shape_w:.2f}, scale={scale_w:.2f})",
        color="coral",
        linewidth=2,
    )
    plt.plot(
        x,
        y_lognorm,
        label=f"LogNormal (μ={mu_ln:.2f}, σ={sigma_ln:.2f})",
        color="seagreen",
        linewidth=2,
    )

    plt.title("Probability Distributions (PDFs)", fontsize=14, fontweight="bold")
    plt.xlabel("X Values", fontsize=12)
    plt.ylabel("Probability Density", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_distributions(multi_results, x_max=300)

In [ ]:
# Create full scenario set
FULL_SCENARIOS = (
    BASE_SCENARIOS
    + [
        {  
            "label": "Weibull (calibrated)",
            "dist": {
                "Type": "Weibull",
                "Shape": float(multi_results["Weibull"].x[0]),
                "Scale": float(multi_results["Weibull"].x[1]),
            },
        },
        {
            "label": "Normal (calibrated)",
            "dist": {
                "Type": "Normal",
                "Mean": float(multi_results["Normal"].x[0]),
                "StdDev": float(multi_results["Normal"].x[1]),
            },
        },
        {
            "label": "LogNormal (calibrated)",
            "dist": {
                "Type": "LogNormal2",
                "Mean": float(multi_results["LogNormal2"].x[0]),
                "StdDev": float(multi_results["LogNormal2"].x[1]),
            },
        },
    ]
)

# Compare the calibrated lifetime scenarios 

In [ ]:
def compare_lifetime_scenarios_fixed(
    scenarios,
    model_time,
    target_stock,
    inflow_data,
    outflow_data,
    negative_inflow_correct=False,
    negative_inflow_style="Default",
):
    """
    Compare multiple lifetime scenarios using flow data.
    """
    comparison_results = []

    # Prepare observed data
    inflow_years = np.array(list(inflow_data.keys()))

    outflow_years = outflow_data["REF_DATE"].to_numpy()
    outflow_values = outflow_data["VALUE"].to_numpy()

    for name, lt_dict in scenarios.items():
        model = dsm.DynamicStockModel(t=model_time, s=target_stock, lt=lt_dict)
        S_C, O_C, I = (
            model.compute_stock_driven_model( 
                NegativeInflowCorrect=negative_inflow_correct
            )
        )
        model.compute_outflow_total()
        # TODO add easier access to dsm_model.remove_cg if negative_inflow_correct is true, to check what was removed (and when)

        year_to_idx = {yr: i for i, yr in enumerate(model_time)}

        # Compare INFLOWS to observed completions
        pred_inflows = np.array(
            [model.i[year_to_idx[yr]] for yr in inflow_years if yr in year_to_idx]
        )
        obs_inflows = np.array(
            [inflow_data[yr] for yr in inflow_years if yr in year_to_idx]
        )

        inflow_metrics = calculate_model_fit_metrics(obs_inflows, pred_inflows)

        # Compare OUTFLOWS to observed demolitions
        valid_outflow_mask = np.isin(outflow_years, list(year_to_idx.keys()))
        pred_outflows = np.array(
            [model.o[year_to_idx[yr]] for yr in outflow_years[valid_outflow_mask]]
        )
        obs_outflows = outflow_values[valid_outflow_mask]

        outflow_metrics = calculate_model_fit_metrics(obs_outflows, pred_outflows)

        # Compile results
        FLOW_WEIGHTS = [ # TODO validate effect/relationship on CALIB_WEIGHTS?
            0.7,
            0.3,
        ]  # Note: 0.7 to acknowledge higher trust in inflow data.
        if sum(FLOW_WEIGHTS) != 1:
            raise ValueError

        metrics = {
            "scenario": name,
            "inflow_rmse": inflow_metrics["rmse"],
            "inflow_nrmse": inflow_metrics["nrmse"],
            "inflow_mape": inflow_metrics["mape"],
            "inflow_r2": inflow_metrics["r_squared"],
            "outflow_rmse": outflow_metrics["rmse"],
            "outflow_nrmse": outflow_metrics["nrmse"],
            "outflow_mape": outflow_metrics["mape"],
            "outflow_r2": outflow_metrics["r_squared"],
            "neg_inflows": np.sum(model.i < 0),
            # Combined score (lower is better)
            "combined_nrmse": FLOW_WEIGHTS[0] * inflow_metrics["nrmse"]
            + FLOW_WEIGHTS[1] * outflow_metrics["nrmse"],
        }

        comparison_results.append(metrics)

    return (
        pd.DataFrame(comparison_results)
        .set_index("scenario")
        .sort_values("combined_nrmse")
    )


In [ ]:
scenarios = {
    item['label']: {
        'Type': item['dist']['Type'],
        **{k: np.full((Nt, 1), v) for k, v in item['dist'].items() if k != 'Type'} # extend to full Nt for ODYM
    }
    for item in FULL_SCENARIOS
}

target_stock = total_stock.to_numpy()

# Run comparison
comparison_df = compare_lifetime_scenarios_fixed(
    scenarios,
    MODEL_TIME,
    target_stock,
    nh_data,
    nh_demolished,
    negative_inflow_correct=NEG_INFLOW_CORRECT,
    negative_inflow_style="Default",  # FIXME UNUSED, TODO use model w/typesplit
)

print("\nScenario Comparison (sorted by combined NRMSE):")
display(comparison_df.round(2))


## Metric Interpretation

| Metric | Good Value | Bad Value | Unit | Interpretation |
|--------|-----------|-----------|--------|----------------|
| `rmse` | Low (→0) | High | dwellings | Absolute error in same units as data |
| `nrmse` | Low (→0%) | High (>50%) | dimensionless | Normalized % error relative to mean |
| `mape` | Low (→0%) | High (>100%) | dimensionless | Average % deviation per observation |
| `r_squared` | High (→1) | Low (<0) | dimensionless | Variance explained; <0 means worse than mean |
| `neg_inflows` | 0 | >0 | item(s) | Number of impossible negative inflows |
| `combined_nrmse` | Low (→0) | High | dimensionless | Overall score (already sorted ascending) |


In [ ]:
# Add interpretation guide after comparison_df
print("\n" + "=" * 60)
print("INTERPRETATION GUIDE")
print("=" * 60)

# Best scenario (already sorted by combined_nrmse - lowest is best)
best_scenario = comparison_df.index[0]
print(f"\n✓ Best overall scenario: {best_scenario}")
print(f"  Combined NRMSE: {comparison_df.loc[best_scenario, 'combined_nrmse']:.1f}%")

# Quality thresholds
print("\nQuality thresholds (NRMSE):")
print("  < 10%  : Excellent fit")
print("  10-25% : Good fit")
print("  25-50% : Acceptable fit")
print("  > 50%  : Poor fit")

# Highlight issues
print("\nScenarios with negative inflows (physically impossible):")
neg_inflow_scenarios = comparison_df[comparison_df["neg_inflows"] > 0]
if len(neg_inflow_scenarios) > 0:
    for idx in neg_inflow_scenarios.index:
        print(
            f"  ⚠ {idx}: {neg_inflow_scenarios.loc[idx, 'neg_inflows']:.0f} years with negative inflows"
        )
else:
    print("  None - all scenarios are physically valid")

# Best for inflows vs outflows separately
best_inflow = comparison_df["inflow_nrmse"].idxmin()
best_outflow = comparison_df["outflow_nrmse"].idxmin()
print(
    f"\nBest for inflow fit: {best_inflow} (NRMSE: {comparison_df.loc[best_inflow, 'inflow_nrmse']:.1f}%)"
)
print(
    f"Best for outflow fit: {best_outflow} (NRMSE: {comparison_df.loc[best_outflow, 'outflow_nrmse']:.1f}%)"
)

# Visual summary
print("\n" + "=" * 60)
print("RANKING (best to worst)")
print("=" * 60)
for i, (idx, row) in enumerate(comparison_df.iterrows(), 1):
    quality = (
        "★★★"
        if row["combined_nrmse"] < 25
        else "★★"
        if row["combined_nrmse"] < 50
        else "★"
    )
    print(f"{i}. {idx:25s} | Combined: {row['combined_nrmse']:5.1f}% | {quality}")

In [ ]:
# Define scenarios
NEG_INFLOW_CORRECT = False
NEG_INFLOW_STYLE = "Default"
scenarios = [scen for scen in FULL_SCENARIOS if (scen['label'] in SUBSET and scen['label']!="Weibull (Deetman)")]

# Run analysis with typesplit method
results_typesplit = run_sensitivity_analysis_typesplit(
    MODEL_TIME,
    total_stock,
    smooth_typesplit,  # updated_tsplit,
    scenarios,
    negative_inflow_correct=NEG_INFLOW_CORRECT,
    negative_inflow_style=NEG_INFLOW_STYLE,  # or 'Default'
)
# NOTE this currently introduces a mistake, as the 'Calibrated' functions are determined on the conventional/basic model, not the typesplit model. does it matter, and if so, how much?

# Plot
fig, ax = plot_sensitivity_results_typesplit(results_typesplit, nh_data, nh_demolished)
if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, 'Fig_calibrated.png')
    plt.savefig(file_path, dpi=DPI)

plt.show()

In [ ]:
scenario_key = "Normal ($\\mu$=150)"

fig, ax = plt.subplots()
# interpolated data
# Total stock in census
# tmp['vintage'] is already based on cohorts_2021, and includes 'total'

df_c = pd.DataFrame(
    dataset[(dataset["type"].isin(TARGET_TYPES)) & (dataset["vintage"] == "1608-2025")]
    .groupby("census_year")["dwellings"]
    .sum()
)

# Decadal stock change
# from the dataset itself, and a simple linear interpolation FIXME hardcoded
df_c.reindex(list(range(1931, 2021))).interpolate(method="linear").diff().plot(
    style="o", ax=ax
)

# from my own model
total_stock.loc[1931:2021].diff().rename("ipfn").plot(
    style=".--", ax=ax
)  # normal, same as df_c


# last 'inflow' calculated by my model
tt = results_typesplit[scenario_key]['inflows'].loc[MODEL_TIME] # retrieve inflows
# tt.index = MODEL_TIME
tt.loc[1968:1999].plot(style=":", ax=ax)

# calculated stock change from completions and demolitions
df_stock_change = df_b.reindex_like(df_a) - df_a
df_stock_change.plot(style=".", ax=ax)

ax.set_title("Stock change, 1968-1999")
ax.set_xlim([1960, 2005])

In [ ]:
# scenarios = [scen for scen in FULL_SCENARIOS if scen['label'] in SUBSET]
scenarios = [scen for scen in FULL_SCENARIOS if (scen['label'] in SUBSET and scen['label']!="Weibull (Deetman)")]

# Create x range for plotting
x = np.linspace(0, 350, 1000)


# Helper function to get scipy distribution object from scenario
def get_scipy_dist(dist_params):
    """Convert distribution parameters to scipy distribution object"""
    dist_type = dist_params["Type"]

    if dist_type == "Weibull":
        return stats.weibull_min(c=dist_params["Shape"], scale=dist_params["Scale"])
    elif dist_type == "Normal":
        return stats.norm(loc=dist_params["Mean"], scale=dist_params["StdDev"])
    elif dist_type == "LogNormal2":
        return stats.lognorm(s=dist_params["StdDev"], scale=np.exp(dist_params["Mean"]))
    else:
        raise ValueError(f"Unknown distribution type: {dist_type}")


# Calculate PDFs and statistics
fig, axs = plt.subplots(2, 2, figsize=(12, 10), dpi=300)
sns.set_theme(context="paper", style="whitegrid", palette=COLOR_PALETTE)

# Panel 1: PDF comparison
ax1 = axs[0, 0]

for scenario in scenarios:
    label = scenario["label"]
    dist = get_scipy_dist(scenario["dist"])
    pdf = dist.pdf(x)
    ax1.plot(x, pdf, label=label, linewidth=2)

ax1.set_xlabel("Lifetime (years)")
ax1.set_ylabel("Probability Density")
ax1.set_title("PDF Comparison")
ax1.legend()
ax1.set_xlim(0, 300)

# Panel 2: Survival Function (what fraction survives past age x)
ax2 = axs[0, 1]

for scenario in scenarios:
    label = scenario["label"]
    dist = get_scipy_dist(scenario["dist"])
    sf = dist.sf(x)
    ax2.plot(x, sf, label=label, linewidth=2)

ax2.set_xlabel("Age (years)")
ax2.set_ylabel("Survival Probability")
ax2.set_title("Survival Function (fraction still standing)")
ax2.legend()
ax2.set_xlim(0, 300)
ax2.set_ylim(0, 1.05)

# Add reference lines
for age in [50, 100, 150, 200]:
    ax2.axvline(age, color="gray", linestyle=":", alpha=0.5)

# Panel 3: CDF (cumulative failures)
ax3 = axs[1, 0]

for scenario in scenarios:
    label = scenario["label"]
    dist = get_scipy_dist(scenario["dist"])
    cdf = dist.cdf(x)
    ax3.plot(x, cdf, label=label, linewidth=2)

ax3.set_xlabel("Age (years)")
ax3.set_ylabel("Cumulative Probability")
ax3.set_title("CDF (fraction demolished by age x)")
ax3.legend()
ax3.set_xlim(0, 300)

# Calculate key statistics for all scenarios
stats_data = {}

for scenario in scenarios:
    label = scenario["label"]
    dist_params = scenario["dist"]
    dist = get_scipy_dist(dist_params)

    # Calculate mode based on distribution type
    if dist_params["Type"] == "Weibull":
        shape = dist_params["Shape"]
        scale = dist_params["Scale"]
        mode = scale * ((shape - 1) / shape) ** (1 / shape) if shape > 1 else 0
        p_x_neg = 0  # Weibull is always ≥0
    elif dist_params["Type"] == "Normal":
        mode = dist_params["Mean"]
        p_x_neg = dist.cdf(0) * 100
    elif dist_params["Type"] == "LogNormal2":
        mu = dist_params["Mean"]
        sigma = dist_params["StdDev"]
        mode = np.exp(mu - sigma**2)
        p_x_neg = 0  # LogNormal is always ≥0
    else:
        mode = np.nan
        p_x_neg = 0

    # Get skewness
    try:
        skew = float(dist.stats(moments="s"))
    except:
        skew = 0 if dist_params["Type"] == "Normal" else np.nan

    stats_data[label] = {
        "Mean": dist.mean(),
        "Median": dist.median(),
        "Mode": mode,
        "Std Dev": dist.std(),
        "CV (%)": dist.std() / dist.mean() * 100,
        "Skewness": skew,
        "P(X<0)": p_x_neg,
        "P(X<50)": dist.cdf(50) * 100,
        "P(X<100)": dist.cdf(100) * 100,
        "P(X>200)": dist.sf(200) * 100,
        "P(X>250)": dist.sf(250) * 100,
        "5th %ile": dist.ppf(0.05),
        "25th %ile": dist.ppf(0.25),
        "50th %ile": dist.ppf(0.50),
        "75th %ile": dist.ppf(0.75),
        "95th %ile": dist.ppf(0.95),
    }

# Create statistics DataFrame
stats_df = pd.DataFrame(stats_data).T
stats_df = stats_df.round(2)

plt.suptitle("Comparison of Lifetime Distributions", fontsize=14, y=1.02)
plt.tight_layout()
if SAVE_FIGS:
    file_path = os.path.join(FIG_DIR, "comparison_pdfs.png")
    plt.savefig(file_path, dpi=DPI)
plt.show()

# Print detailed statistics
print("=" * 80)
print("DETAILED DISTRIBUTION STATISTICS")
print("=" * 80)
print(stats_df.T.to_string())

print("\n" + "=" * 80)
print("PHYSICAL PLAUSIBILITY CHECK")
print("=" * 80)

for name, data in stats_data.items():
    print(f"\n{name}:")
    if data["P(X<0)"] > 0:
        print(f"  ⚠️  WARNING: {data['P(X<0)']:.2f}% probability of negative lifetimes!")
    else:
        print("  ✓  No negative values possible")

    if data["P(X<50)"] > 5:
        print(
            f"  ⚠️  High early failures: {data['P(X<50)']:.1f}% demolished before 50 years"
        )
    else:
        print(
            f"  ✓  Low early failures: {data['P(X<50)']:.1f}% demolished before 50 years"
        )

    if data["P(X>250)"] > 5:
        print(f"  ⚠️  Long tail: {data['P(X>250)']:.1f}% survive past 250 years")
    else:
        print(
            f"  ✓  Reasonable upper bound: {data['P(X>250)']:.1f}% survive past 250 years"
        )